## Code Review: `btcusdt_utils` Package
**Reviewer**: Senior ML Ops Engineer | **Date**: 2026-04-19 | **Verdict**: ✅ Approve with comments

---

### Executive Summary

Well-architected utility package with clean module boundaries, correct AFML implementations, and strong Polars/NumPy idioms. The primary risks are **performance bottlenecks in Python-loop hot paths**, a **missing early-stopping bug in training**, and **hardcoded environment config**. No blockers — all findings are addressable incrementally.

| Dimension | Rating | Notes |
|---|---|---|
| Architecture | ★★★★★ | Clean separation of concerns, no circular deps, schema-adaptive features |
| Correctness | ★★★★☆ | One functional bug (training.py early stopping), minor edge cases |
| Performance | ★★★☆☆ | Four Python-loop hot paths that will bottleneck at scale |
| Code Style | ★★★★★ | Consistent Fluent Python patterns, frozen dataclasses, proper `__repr__` |
| Documentation | ★★★★★ | Excellent docstrings, AFML references, algorithm descriptions |
| Testability | ★★☆☆☆ | Zero test files — significant gap for a shared utility package |

---

### 🔴 Critical (1 finding)

**[T1] `training.py` — LightGBM/XGBoost train without early stopping**

`create_lgb_objective` and `create_xgb_objective` call `model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)])` but pass **no early stopping callback**. LightGBM trains all `n_estimators` (up to 500) regardless of validation plateau. This wastes compute and risks overfitting — the Optuna study optimizes over a parameter that isn't being exploited.

```python
# Current (broken)
model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)])

# Fix
model.fit(
    X_tr, y_tr,
    eval_set=[(X_v, y_v)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
```

Same pattern for XGBoost — add `early_stopping_rounds=50` or the equivalent callback.

---

### 🟡 High Priority (5 findings)

**[C1] `constants.py` — Hardcoded catalog name**

`CATALOG = "smart-trading-dev"` bakes in a dev-environment reference. When promoting to staging/prod, every import silently points at dev. Use an env var with fallback:

```python
import os
CATALOG = os.environ.get("BTCUSDT_CATALOG", "smart-trading-dev")
```

**[C2] `constants.py` — SQL dialect coupling in paths**

`RAW_AGGTRADES_PATH` uses backtick quoting (`` `catalog`.`schema`.`table` ``), coupling the constant to Spark SQL syntax. Store unquoted three-part names and quote at point of use.

**[P1] `bars.py` → `cusum_filter` — Pure Python loop over tick data**

CUSUM filter iterates tick-by-tick in Python. At 52M ticks, this is a minutes-long bottleneck. The stateful nature prevents pure NumPy vectorization, but **Numba `@njit`** would bring it to near-C speed with zero API change.

**[P2] `stationarity.py` → `ffd` — Python loop with `np.dot` per step**

The FFD convolution is `O(n × w)` with Python-level iteration. Replace with `np.convolve(series, weights[::-1], mode='full')[:n]` for a single vectorized pass, or `scipy.signal.fftconvolve` for large windows.

**[P3] `quality.py` → `label_avg_uniqueness` — Double Python loop**

Both concurrency counting and per-sample uniqueness use explicit Python `for` loops. Concurrency can be vectorized with `np.add.at(concurrency, range(s, e+1), 1)` per event, and uniqueness with `np.mean(1.0 / concurrency[s:e+1])` is already semi-vectorized but the outer loop remains. Consider Numba.

---

### 🟢 Medium Priority (8 findings)

**[S1] `__init__.py` — Exports underscore-prefixed names**

Color constants `_BG`, `_RED`, etc. are exported in `__all__`. Leading underscores conventionally signal private API. Either drop the underscores (they're clearly public) or keep them private and export only `DARK_LAYOUT` and `dark_fig`.

**[S2] `features.py` — `FEATURE_COLS` is a mutable list**

Following your Fluent Python rules: *"Prefer tuple for fixed-length... and immutable sequences; they use less memory and signal immutability."* Change to `tuple[str, ...]`.

**[S3] `stationarity.py` → `find_min_ffd` — DRY violation**

Calls `adfuller` directly instead of reusing its own `adf_test` function. Also returns `list[dict]` instead of a dataclass — inconsistent with the rest of the codebase.

**[S4] `stationarity.py` — Hardcoded significance thresholds**

`adf_test` hardcodes `p < 0.01` and `kpss_test` hardcodes `p > 0.05`. Make these keyword-only parameters with defaults.

**[S5] `features.py` — `bb_pct` division-by-zero**

When price is flat within the Bollinger window, `bb_upper == bb_lower`, causing `0/0`. Add a guard: `.fill_nan(0.5)` or use `pl.when(denom > 0).then(...).otherwise(0.5)`.

**[S6] `evaluation.py` — `compute_metrics` rounds internally**

`round(acc, 5)` inside a utility function forces precision on all callers. Return raw floats; let callers format.

**[S7] `evaluation.py` — `build_comparison_table` accepts loose typing**

Signature uses `dict[str, dict[str, Any]]` when it could accept `dict[str, TrainResult]` directly, leveraging the dataclass already defined in `training.py`.

**[S8] `training.py` — Global warning suppression**

`warnings.filterwarnings("ignore", category=UserWarning)` is too broad and persists beyond the function. Scope with a `warnings.catch_warnings()` context manager and filter only Optuna/LightGBM warnings.

---

### 💡 Recommendations

**[R1] Add a `tests/` directory** — Even minimal property-based tests (e.g., `compute_bar_quality` returns valid ranges, `apply_triple_barrier` labels sum to n_events, feature count matches `FEATURE_COLS`) would catch regressions. Use `pytest` + `hypothesis` for numeric edge cases.

**[R2] Add `__version__`** — The package has no version identifier. Add `__version__ = "0.1.0"` in `__init__.py` for reproducibility tracking in MLflow.

**[R3] `labeling.py` → Return a NamedTuple** — The 5-tuple return from `apply_triple_barrier` is opaque. Following Fluent Python: *"Use typing.NamedTuple for immutable records."*

```python
class TripleBarrierResult(NamedTuple):
    labels: np.ndarray
    touch_bars: np.ndarray
    touch_types: np.ndarray
    event_starts: np.ndarray
    event_ends: np.ndarray
```

**[R4] `training.py` → Use nested MLflow runs** — Currently all 5 models log as sibling runs. Use a parent run for the experiment and child runs per model for cleaner experiment tracking.

**[R5] `training.py` → Add Optuna pruning** — `MedianPruner` with LightGBM/XGBoost integration callbacks would halve tuning time by killing underperforming trials early.

**[R6] `bars.py` → `build_time_bars` missing gap handling** — The notebook plan (Cell 7) specifies forward-fill for missing minutes, but `build_time_bars` doesn't implement it. Either add an `ffill_gaps=True` parameter or document that gap handling is the caller's responsibility.

**[R7] `bars.py` → `build_threshold_bars` silently drops tail ticks** — Ticks after the last bar boundary are truncated without warning. Log or return metadata about dropped rows.

---

### What's Done Well

* **Frozen dataclasses with proper `__repr__`** throughout (`BarQuality`, `LabelQuality`, `BaselineResult`, `ModelConfig`, `TrainResult`) — textbook Fluent Python
* **Schema-adaptive feature computation** — `compute_technical_indicators` conditionally computes microstructure features based on available columns, making it reusable across bar types
* **Vectorized wavefront scan** in `labeling.py` — elegant O(max_hold × n) algorithm avoiding per-event Python loops
* **Prefix-sum + searchsorted** in `build_threshold_bars` — \~100x faster than naive accumulation loops
* **Polars lazy pipeline** in `features.py` — single `.collect()` for all 39 indicators
* **5 baseline strategies** in `evaluation.py` — sets proper context for model evaluation
* **Consistent `from __future__ import annotations`** — future-proofed type hints across all modules

## AutoML: Predict Next Price Direction (BTCUSDT)

### Data Summary
| Metric | Value |
|---|---|
| Symbol | BTCUSDT |
| Date Range | 2026-02-01 → 2026-02-28 (27 days) |
| Total Rows | **52,474,665** (tick-level aggregated trades) |
| Columns | price, quantity, is_buyer_maker, transact_timestamp, etc. |

### Strategy for 52M Rows
> **Raw tick data → 1-min OHLCV bars** reduces 52M rows to \~40K rows.  
> Aggregation happens in **Spark** (distributed). Feature engineering uses **Polars** (faster than Pandas, zero-copy from Arrow).  
> AutoML runs on the small aggregated + feature-engineered dataset.

---

### Restructured Notebook Plan (EDA-First Approach)

Follows the **Databricks MLOps workflow**: EDA → Preprocessing → Feature Engineering → Training → Evaluation → Registration.  
EDA happens in **three phases** — raw data, OHLCV bars, and engineered features — so every transformation is *informed* and *validated* by data understanding.

---

#### PHASE 1: ENVIRONMENT SETUP
| Cell | Step | Description |
|---|---|---|
| **8** | **Install Dependencies** | `%pip install polars`, restart kernel |
| **9** | **Re-init Variables + Imports** | Redefine constants (CATALOG, BRONZE, etc.), import polars/spark/mlflow/matplotlib |

#### PHASE 2: RAW DATA EDA (Before Any Transformation)
| Cell | Step | Description |
|---|---|---|
| **10** | **Data Profiling** | Schema, shape, describe(), dtypes, null counts per column, basic stats (price range, quantity range, timestamp range). All in Spark. |
| **11** | **Raw Data EDA — Visualizations** | Price time-series plot, trades-per-minute histogram, quantity distribution, buyer/maker ratio over time, hourly/daily trade volume patterns, time gap analysis (gaps > 1 min). These visuals reveal seasonality, outliers, and gaps that directly affect OHLCV aggregation. |
| **12** | **Data Quality Assessment** | Summarize EDA findings: document identified gaps, null patterns, outliers, duplicates. Record explicit decisions for the aggregation step (e.g., how to handle gaps, whether to filter outlier quantities). This cell is a markdown + code checkpoint. |

#### PHASE 3: DATA TRANSFORMATION (Informed by EDA)
| Cell | Step | Description |
|---|---|---|
| **13** | **Aggregate → 1-min OHLCV Bars (Spark)** | Spark-distributed aggregation using struct-ordering for deterministic open/close. Handles missing minutes with forward-fill based on gap analysis from Phase 2. Converts to Polars via Arrow. |
| **14** | **OHLCV Validation & EDA** | Verify aggregation integrity: row count (\~40K), open\[i+1\] ≈ close\[i\], no nulls in OHLCV columns, candlestick sample visualization, distribution of bar returns, volume distribution, gap-filled bars flagged. This catches aggregation bugs before feature engineering. |

#### PHASE 4: FEATURE ENGINEERING (Informed by OHLCV EDA)
| Cell | Step | Description |
|---|---|---|
| **15** | **Feature Engineering — Technical Indicators (Polars)** | SMA, EMA, RSI, MACD, Bollinger Bands, ATR, price returns, volume features, candle features. Window sizes informed by trade density findings from Phase 2. |
| **16** | **Target Variable** | Binary label: next bar close > current close → 1 (UP), else 0 (DOWN). Sanity check for look-ahead bias. |

#### PHASE 5: FEATURE EDA & VALIDATION (Before Modeling)
| Cell | Step | Description |
|---|---|---|
| **17** | **Feature EDA — Distributions & Quality** | Feature type table (name + dtype), null analysis (warm-up period), infinite value check, histograms of key features (RSI, MACD, returns, volume ratio, bb_pct, atr_pct), outlier detection. |
| **18** | **Feature EDA — Correlations & Target** | Correlogram (feature-feature heatmap), identify pairs with \|r\| > 0.95 for removal, feature-target correlation bar chart (top 20), class balance bar chart + percentages, **data leakage check** (any feature \|r\| > 0.5 with target = investigate), time-series plot with overlaid indicators. |

#### PHASE 6: DATA PREPARATION
| Cell | Step | Description |
|---|---|---|
| **19** | **Prepare Final Dataset** | Drop warm-up nulls, handle inf, remove highly correlated features (decisions from Cell 18), select features + target, cast target to string, time-based 80/20 split, save to Delta table in Silver layer. |
| **20** | **Establish Baselines** | Compute naive baselines on the test split: majority-class accuracy, random-coin accuracy, "predict last direction" accuracy, "always UP" accuracy. These are the bars AutoML MUST beat. Without baselines, a 52% accuracy looks good but may be worse than a coin flip. |

#### PHASE 7: MODEL TRAINING
| Cell | Step | Description |
|---|---|---|
| **21** | **Run Databricks AutoML** | `automl.classify()` with `time_col` for chronological split, `primary_metric="roc_auc"`. AutoML tries LightGBM, XGBoost, Logistic Regression, Decision Tree, Random Forest with automated hyperparameter tuning. |

#### PHASE 8: EVALUATION & DEPLOYMENT
| Cell | Step | Description |
|---|---|---|
| **22** | **Evaluate Best Model** | Confusion matrix, ROC-AUC, PR-AUC, classification report, feature importances, prediction probability distributions, time-series performance plot (rolling accuracy), **compare vs baselines from Cell 20**. |
| **23** | **Register Model to Unity Catalog** | Set UC registry, register with signature + input example, add description/tags (symbol, interval, task), set "champion" alias, quick inference test on sample rows. |

---

### What Changed vs. Previous Plan
| Issue | Before | After |
|---|---|---|
| **EDA placement** | Step 7 (after feature engineering) | **Steps 10-12** (before any transformation) |
| **EDA phases** | 1 monolithic step | **3 phases**: raw data, OHLCV, features |
| **Aggregation validation** | None | **Step 14**: verify OHLCV integrity |
| **Baseline models** | None | **Step 20**: naive baselines to contextualize AutoML |
| **EDA → decisions flow** | EDA validated after the fact | EDA **informs** aggregation (gaps), features (window sizes), and prep (correlation removal) |
| **Data quality checkpoint** | None | **Step 12**: explicit documentation of findings and decisions |

### Key Design Decisions
* **EDA-first, always**: Databricks MLOps workflow mandates EDA before preprocessing. Every transformation is informed by data understanding.
* **Polars over Pandas**: \~5-10x faster for columnar ops, native lazy evaluation, zero-copy Arrow interchange with Spark.
* **1-minute bars**: \~40K data points balances noise smoothing with sufficient training data.
* **Time-based split (not random)**: prevents look-ahead bias in financial time series.
* **Baselines before AutoML**: a 52% model is useless if majority-class gets 51%. Baselines set the bar.
* **Spark for aggregation, Polars for features**: distributed compute where data is large, fast local compute where it's small.

In [0]:
# Install ALL dependencies upfront (single kernel restart)
# btcusdt_utils needs: polars, statsmodels, plotly, scikit-learn
# Phase 7B (training) needs: lightgbm, xgboost, optuna
%pip install polars lightgbm xgboost optuna statsmodels plotly scikit-learn --quiet
%restart_python

In [0]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Single Source of Truth (REFACTORED)
# ═══════════════════════════════════════════════════════════════
# After %restart_python, all variables are lost.
# This cell uses the NEW btcusdt_utils.config module to centralize
# all constants, hyperparameters, and paths.
#
# MIGRATION: Old scattered constants → NotebookConfig dataclass
# ═══════════════════════════════════════════════════════════════

import os
import sys
import importlib

# ── Force reload btcusdt_utils (picks up new config.py) ─────────────
if 'btcusdt_utils' in sys.modules:
    import btcusdt_utils
    importlib.reload(btcusdt_utils)
    print("✓ Reloaded btcusdt_utils (now includes config.py)\n")

# ── Import btcusdt_utils (config + viz theme) ────────────────────
from btcusdt_utils import NotebookConfig
from btcusdt_utils import (
    _BG, _BG_CARD, _TEXT, _TEXT_MUT, _GRID,
    _RED, _GREEN, _BLUE, _ORANGE, _PURPLE, _CYAN,
    _FONT, DARK_LAYOUT, dark_fig,
)

# ── Third-party imports ──────────────────────────────────────────
import polars as pl
import numpy as np
import mlflow
import plotly.graph_objects as go
from pyspark.sql import functions as F, Window

# ════════════════════════════════════════════════════════════════
# NEW APPROACH: All config in one immutable object
# ════════════════════════════════════════════════════════════════

# Default configuration (uses all sensible defaults)
cfg = NotebookConfig()

# Custom configuration example (uncomment to override):
# cfg = NotebookConfig(
#     train_ratio=0.70,
#     optuna_trials=50,
#     optuna_timeout_sec=900,
#     top_n_features=30,
#     experiment_suffix="_v2",
# )

# ── Backward compatibility: extract to module-level variables ────
# This allows existing cells that reference CATALOG, TRAIN_RATIO, etc.
# to continue working without modification.

CATALOG = cfg.catalog.catalog
BRONZE = cfg.catalog.bronze
SILVER = cfg.catalog.silver
GOLD = cfg.catalog.gold
VOLUME_SCHEMA = cfg.catalog.volume_schema

AGGTRADES_TABLE = cfg.catalog.aggtrades_table
NEWS_TABLE = cfg.catalog.news_table
AGGTRADES_PATH = cfg.aggtrades_path
NEWS_PATH = cfg.news_path

TRAIN_RATIO = cfg.training.train_ratio
VAL_RATIO = cfg.training.val_ratio
TEST_RATIO = cfg.training.test_ratio
OPTUNA_TRIALS = cfg.training.optuna_trials
OPTUNA_TIMEOUT_SEC = cfg.training.optuna_timeout_sec

TOP_N_FEATURES = cfg.viz.top_n_features
ROLLING_WINDOW_SIZE = cfg.viz.rolling_window_size

EXPERIMENT_NAME = cfg.experiment_name

# ── Set MLflow experiment ────────────────────────────────────────
mlflow.set_experiment(EXPERIMENT_NAME)

# ── Print summary ────────────────────────────────────────────────
cfg.print_summary()

print("\nDependencies:")
print(f"  Polars:         {pl.__version__}")
print(f"  NumPy:          {np.__version__}")
print(cfg.section_divider)
print("\n✓ Configuration loaded from NotebookConfig")
print("  All downstream cells can use cfg.* or backward-compatible")
print("  module-level variables (CATALOG, TRAIN_RATIO, etc.)")
print(cfg.section_divider)

In [0]:
# PHASE 2A — Data Profiling (Spark — Full 52M Rows)
# ============================================================

df_raw = spark.table(AGGTRADES_PATH)

# ── 1. SCHEMA INSPECTION ──────────────────────────────────────
print("=" * 60)
print("1. SCHEMA INSPECTION")
print("=" * 60)
df_raw.printSchema()
print(f"Columns ({len(df_raw.columns)}): {df_raw.columns}")

# ── 2-4. CORE STATS (single distributed pass) ────────────────
# Combine row count, distinct IDs, timestamp range, and price
# range into one aggregation to avoid multiple 52M-row scans.
print("\n" + "=" * 60)
print("2-4. CORE STATISTICS (single pass)")
print("=" * 60)

core_stats_df = df_raw.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("agg_trade_id").alias("distinct_ids"),
    F.min("transact_timestamp").alias("min_ts"),
    F.max("transact_timestamp").alias("max_ts"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("avg_price"),
    F.stddev("price").alias("stddev_price"),
    F.sum(F.when(F.col("price") <= 0, 1).otherwise(0)).alias("non_positive_prices"),
)
display(core_stats_df)

# Extract scalars for downstream cells
core_stats = core_stats_df.first()
total_rows = core_stats["total_rows"]
distinct_ids = core_stats["distinct_ids"]
duplicate_count = total_rows - distinct_ids
min_ts = core_stats["min_ts"]
max_ts = core_stats["max_ts"]
time_span_days = (max_ts - min_ts).total_seconds() / 86400
min_price = float(core_stats["min_price"])
max_price = float(core_stats["max_price"])
avg_price = float(core_stats["avg_price"])
stddev_price = float(core_stats["stddev_price"])

print(f"\nDuplicates: {duplicate_count:,} ({total_rows:,} total − {distinct_ids:,} distinct)")
print(f"Time span:  {time_span_days:.2f} days")

# ── 5. QUANTITY DISTRIBUTION (percentiles) ────────────────────
print("\n" + "=" * 60)
print("5. QUANTITY DISTRIBUTION")
print("=" * 60)

qty_stats_df = df_raw.select(
    F.min("quantity").alias("min"),
    F.percentile_approx(F.col("quantity").cast("double"), 0.25).alias("p25"),
    F.percentile_approx(F.col("quantity").cast("double"), 0.50).alias("p50"),
    F.avg("quantity").alias("avg"),
    F.percentile_approx(F.col("quantity").cast("double"), 0.75).alias("p75"),
    F.percentile_approx(F.col("quantity").cast("double"), 0.95).alias("p95"),
    F.percentile_approx(F.col("quantity").cast("double"), 0.99).alias("p99"),
    F.max("quantity").alias("max"),
)
display(qty_stats_df)

# Store for Phase 2C decisions
qty_p99 = float(qty_stats_df.first()["p99"])

# ── 6. NULL CHECK (all columns) ───────────────────────────────
print("\n" + "=" * 60)
print("6. NULL CHECK (per column)")
print("=" * 60)

null_df = df_raw.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]
)
display(null_df)

# ── 7. DTYPE VERIFICATION ────────────────────────────────────
print("\n" + "=" * 60)
print("7. DTYPE VERIFICATION")
print("=" * 60)

expected_types = {
    "agg_trade_id": "bigint", "price": "double",
    "quantity": "double", "first_trade_id": "bigint",
    "last_trade_id": "bigint", "transact_time": "bigint",
    "is_buyer_maker": "boolean", "is_best_match": "boolean",
    "symbol": "string", "transact_timestamp": "timestamp",
    "trade_date": "date",
}

dtype_rows = [
    (f.name, expected_types.get(f.name, "?"), f.dataType.simpleString(),
     "✓" if f.dataType.simpleString() == expected_types.get(f.name, "?") else "✗ MISMATCH")
    for f in df_raw.schema.fields
]
dtype_df = spark.createDataFrame(dtype_rows, ["column", "expected", "actual", "status"])
display(dtype_df)

# ── DESCRIBE (full numeric summary) ──────────────────────────
print("\n" + "=" * 60)
print("FULL NUMERIC SUMMARY (describe)")
print("=" * 60)
display(df_raw.describe())

In [0]:
# PHASE 2B — Raw Data EDA Visualizations
# ============================================================
# All aggregations run distributed in Spark (52M rows).
# Only small aggregated results collected for Plotly.
# Variables from Cell 10: df_raw, total_rows, qty_p99,
#   min_ts, max_ts, min_price, max_price, duplicate_count
# ============================================================

import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

PLOTLY_TEMPLATE = "plotly_white"
COLORS = {
    "price": "#1f77b4",
    "trades": "#2ca02c",
    "quantity": "#9467bd",
    "buyer": "#d62728",
    "gap": "#ff7f0e",
}

# ═══════════════════════════════════════════════════════════════
# SPARK AGGREGATIONS (all run first, then plot)
# ═══════════════════════════════════════════════════════════════

# ── A. Hourly: price, volume, buyer ratio, trade count ────────
print("▸ Aggregating hourly stats...")
hourly_pdf = (
    df_raw
    .withColumn("hour_bucket", F.date_trunc("hour", "transact_timestamp"))
    .groupBy("hour_bucket")
    .agg(
        F.avg(F.col("price").cast("double")).alias("avg_price"),
        F.count("*").alias("trade_count"),
        F.avg(F.col("is_buyer_maker").cast("int")).alias("buyer_maker_ratio"),
        F.sum(F.col("quantity").cast("double")).alias("total_volume"),
    )
    .orderBy("hour_bucket")
    .toPandas()
)
print(f"  → {len(hourly_pdf)} hourly buckets")

# ── B. Trades per minute ─────────────────────────────────────────
print("▸ Aggregating trades per minute...")
tpm_pdf = (
    df_raw
    .withColumn("min_bucket", F.date_trunc("minute", "transact_timestamp"))
    .groupBy("min_bucket")
    .agg(F.count("*").alias("trade_count"))
    .orderBy("min_bucket")
    .toPandas()
)
print(f"  → {len(tpm_pdf)} minute buckets")

# ── C. Heatmap: hour_of_day × day_of_month ───────────────────────
print("▸ Aggregating heatmap data...")
heatmap_pdf = (
    df_raw
    .withColumn("hour_of_day", F.hour("transact_timestamp"))
    .withColumn("day_of_month", F.dayofmonth("trade_date"))
    .groupBy("day_of_month", "hour_of_day")
    .agg(F.count("*").alias("trade_count"))
    .toPandas()
)

# ── D. Quantity sample (100K rows) ───────────────────────────
print("▸ Sampling quantity distribution...")
qty_pdf = (
    df_raw
    .select(F.col("quantity").cast("double").alias("quantity"))
    .sample(fraction=min(200_000 / total_rows, 1.0), seed=42)
    .limit(100_000)
    .toPandas()
)
print(f"  → {len(qty_pdf)} samples")

# ── E. Daily summary ─────────────────────────────────────────
print("▸ Aggregating daily summary...")
daily_pdf = (
    df_raw
    .groupBy("trade_date")
    .agg(
        F.count("*").alias("trade_count"),
        F.min(F.col("price").cast("double")).alias("min_price"),
        F.max(F.col("price").cast("double")).alias("max_price"),
        (F.max(F.col("price").cast("double")) - F.min(F.col("price").cast("double"))).alias("daily_range"),
        F.sum(F.col("quantity").cast("double")).alias("total_volume"),
        F.avg(F.col("is_buyer_maker").cast("int")).alias("buyer_ratio"),
    )
    .orderBy("trade_date")
    .toPandas()
)
print(f"  → {len(daily_pdf)} days")

# ═══════════════════════════════════════════════════════════════
# TIME GAP ANALYSIS (minute-level — avoids LAG over 52M rows)
# ═══════════════════════════════════════════════════════════════
print("\n▸ Analyzing time gaps...")
full_minutes = pd.date_range(
    start=tpm_pdf["min_bucket"].min(),
    end=tpm_pdf["min_bucket"].max(),
    freq="min",
)
existing_minutes = set(tpm_pdf["min_bucket"])
missing_minutes = sorted(m for m in full_minutes if m not in existing_minutes)

total_expected_bars = len(full_minutes)
actual_bars = len(tpm_pdf)
gap_count = len(missing_minutes)
gap_pct = 100.0 * gap_count / total_expected_bars if total_expected_bars else 0.0

print(f"  Expected 1-min bars: {total_expected_bars:,}")
print(f"  Actual bars:         {actual_bars:,}")
print(f"  Missing minutes:     {gap_count:,} ({gap_pct:.2f}%)")

# ═══════════════════════════════════════════════════════════════
# VIZ 1: Price Time-Series (hourly avg) — Plotly
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("VIZ 1: PRICE TIME-SERIES")
print("=" * 70)

fig1 = px.line(
    hourly_pdf, x="hour_bucket", y="avg_price",
    title="BTCUSDT Hourly Average Price — Feb 2026",
    labels={"hour_bucket": "Date", "avg_price": "Price (USD)"},
    template=PLOTLY_TEMPLATE,
)
fig1.update_traces(
    line=dict(color=COLORS["price"], width=1.2),
    hovertemplate="%{x|%b %d %H:%M}<br>$%{y:,.2f}<extra></extra>",
)
fig1.update_layout(
    width=1200, height=450,
    yaxis_tickformat="$,.0f",
    xaxis=dict(
        dtick=2 * 86_400_000,  # 2 days in ms
        tickformat="%b %d",
        rangeslider=dict(visible=True, thickness=0.05),
    ),
    margin=dict(l=60, r=30, t=50, b=40),
)
display(fig1)

# ═══════════════════════════════════════════════════════════════
# VIZ 2: Trades-per-Minute Distribution — Plotly
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("VIZ 2: TRADES-PER-MINUTE DISTRIBUTION")
print("=" * 70)
tpm_counts = tpm_pdf["trade_count"]

fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=tpm_counts, nbinsx=100,
    marker_color=COLORS["trades"],
    opacity=0.85,
    name="Trades/min",
))
fig2.add_vline(
    x=float(tpm_counts.median()),
    line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"Median: {tpm_counts.median():,.0f}",
    annotation_position="top right",
    annotation_font_color="red",
)
fig2.add_vline(
    x=float(tpm_counts.mean()),
    line_dash="dash", line_color="orange", line_width=2,
    annotation_text=f"Mean: {tpm_counts.mean():,.0f}",
    annotation_position="top left",
    annotation_font_color="orange",
)
fig2.update_layout(
    title="Distribution of Trades per Minute",
    xaxis_title="Trades per Minute",
    yaxis_title="Frequency (# of minutes)",
    template=PLOTLY_TEMPLATE,
    width=1100, height=450,
    showlegend=False,
    margin=dict(l=60, r=30, t=50, b=40),
)
display(fig2)

print(f"\nTrades-per-Minute Statistics:")
print(f"  Min:    {tpm_counts.min():>10,}")
print(f"  P5:     {tpm_counts.quantile(0.05):>10,.0f}")
print(f"  P25:    {tpm_counts.quantile(0.25):>10,.0f}")
print(f"  Median: {tpm_counts.median():>10,.0f}")
print(f"  Mean:   {tpm_counts.mean():>10,.0f}")
print(f"  P75:    {tpm_counts.quantile(0.75):>10,.0f}")
print(f"  P95:    {tpm_counts.quantile(0.95):>10,.0f}")
print(f"  Max:    {tpm_counts.max():>10,}")
low_activity = (tpm_counts < 10).sum()
print(f"  Minutes with < 10 trades: {low_activity:,}")

# ═══════════════════════════════════════════════════════════════
# VIZ 3: Trades-per-Hour Heatmap (Hour × Day) — Plotly
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("VIZ 3: TRADE ACTIVITY HEATMAP")
print("=" * 70)
pivot = heatmap_pdf.pivot_table(
    values="trade_count", index="hour_of_day",
    columns="day_of_month", fill_value=0,
)

fig3 = px.imshow(
    pivot,
    color_continuous_scale="sunset",
    aspect="auto",
    title="Trade Activity Heatmap — Hour of Day (UTC) × Day of Month",
    labels=dict(
        x="Day of Month (February 2026)",
        y="Hour of Day (UTC)",
        color="Trade Count",
    ),
)
fig3.update_layout(
    template=PLOTLY_TEMPLATE,
    width=1200, height=600,
    margin=dict(l=60, r=30, t=50, b=40),
)
display(fig3)

# ═══════════════════════════════════════════════════════════════
# VIZ 4: Quantity Distribution (log-log) — Plotly
# Pre-compute log-spaced bins with numpy, then render as go.Bar
# (Plotly histograms don't natively support log-x binning)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("VIZ 4: QUANTITY DISTRIBUTION")
print("=" * 70)

qty_vals = qty_pdf["quantity"]
qty_positive = qty_vals[qty_vals > 0].values

# Geometrically-spaced bins for correct log-log rendering
bin_edges = np.geomspace(qty_positive.min(), qty_positive.max(), num=201)
counts, edges = np.histogram(qty_positive, bins=bin_edges)
bin_centers = np.sqrt(edges[:-1] * edges[1:])  # geometric mean

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    x=bin_centers, y=counts,
    marker_color=COLORS["quantity"],
    opacity=0.85,
    name="Quantity",
    hovertemplate="Qty: %{x:.5f} BTC<br>Count: %{y:,}<extra></extra>",
))
fig4.add_vline(
    x=qty_p99, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"P99: {qty_p99:.5f} BTC",
    annotation_position="top right",
    annotation_font_color="red",
)
fig4.update_layout(
    title="Quantity Distribution (Log-Log Scale, 100K sample)",
    xaxis_title="Quantity (BTC) — log scale",
    yaxis_title="Frequency — log scale",
    xaxis_type="log",
    yaxis_type="log",
    template=PLOTLY_TEMPLATE,
    width=1100, height=450,
    showlegend=False,
    margin=dict(l=60, r=30, t=50, b=40),
)
display(fig4)

# ═══════════════════════════════════════════════════════════════
# VIZ 5: Buyer/Maker Ratio Over Time — Plotly
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("VIZ 5: BUYER/MAKER RATIO")
print("=" * 70)

fig5 = go.Figure()

# Invisible baseline at 0.5 (anchor for fill="tonexty")
fig5.add_trace(go.Scatter(
    x=hourly_pdf["hour_bucket"],
    y=[0.5] * len(hourly_pdf),
    mode="lines",
    line=dict(color="gray", width=1, dash="dot"),
    name="50% (balanced)",
    showlegend=True,
    hoverinfo="skip",
))

# Buyer-maker ratio with fill between ratio and 0.5 baseline
fig5.add_trace(go.Scatter(
    x=hourly_pdf["hour_bucket"],
    y=hourly_pdf["buyer_maker_ratio"],
    mode="lines",
    line=dict(color=COLORS["buyer"], width=1),
    fill="tonexty",
    fillcolor="rgba(214, 39, 40, 0.12)",
    name="Buyer-Maker Ratio",
    hovertemplate="%{x|%b %d %H:%M}<br>Ratio: %{y:.4f}<extra></extra>",
))

fig5.update_layout(
    title="Buyer-is-Maker Ratio Over Time (Hourly)",
    xaxis_title="Date",
    yaxis_title="Buyer-Maker Ratio",
    yaxis_range=[0.30, 0.70],
    xaxis=dict(dtick=2 * 86_400_000, tickformat="%b %d"),
    template=PLOTLY_TEMPLATE,
    width=1200, height=450,
    legend=dict(x=0.85, y=0.98),
    margin=dict(l=60, r=30, t=50, b=40),
)
display(fig5)

overall_buyer_ratio = hourly_pdf["buyer_maker_ratio"].mean()
print(f"\nBuyer-Maker Ratio Summary:")
print(f"  Overall mean:  {overall_buyer_ratio:.4f} ({overall_buyer_ratio:.1%})")
print(f"  Min (hourly):  {hourly_pdf['buyer_maker_ratio'].min():.4f}")
print(f"  Max (hourly):  {hourly_pdf['buyer_maker_ratio'].max():.4f}")
print(f"  Std (hourly):  {hourly_pdf['buyer_maker_ratio'].std():.4f}")

# ═══════════════════════════════════════════════════════════════
# VIZ 6: Time Gap Analysis — Plotly
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("VIZ 6: TIME GAP ANALYSIS")
print("=" * 70)
if gap_count > 0:
    gap_ser = pd.Series(missing_minutes)
    gap_starts = gap_ser.diff() > pd.Timedelta(minutes=1)
    streak_ids = gap_starts.cumsum()
    gap_streaks = (
        pd.DataFrame({"minute": missing_minutes, "streak": streak_ids})
        .groupby("streak")
        .agg(
            start=("minute", "first"),
            end=("minute", "last"),
            duration_min=("minute", "count"),
        )
        .sort_values("duration_min", ascending=False)
    )
    print(f"  Distinct gap events: {len(gap_streaks)}")
    print(f"  Longest gap:         {gap_streaks['duration_min'].max()} min")
    print(f"  Mean gap length:     {gap_streaks['duration_min'].mean():.1f} min")
    print(f"\n  Top 10 longest gaps:")
    display(gap_streaks.head(10).reset_index(drop=True))

    fig6 = px.histogram(
        gap_streaks, x="duration_min", nbins=50,
        title=(
            f"Gap Streak Length Distribution "
            f"({gap_count:,} missing min, {len(gap_streaks)} events)"
        ),
        labels={
            "duration_min": "Gap Duration (consecutive missing minutes)",
            "count": "Frequency",
        },
        color_discrete_sequence=[COLORS["gap"]],
        template=PLOTLY_TEMPLATE,
    )
    fig6.update_layout(
        width=1000, height=450,
        yaxis_title="Frequency",
        margin=dict(l=60, r=30, t=50, b=40),
    )
    display(fig6)
else:
    gap_streaks = pd.DataFrame(columns=["start", "end", "duration_min"])
    print("  ✓ No missing minutes — every minute has ≥1 trade.")

# ═══════════════════════════════════════════════════════════════
# VIZ 7: Daily Summary Table (already uses display — unchanged)
# ═══════════════════════════════════════════════════════════════
daily_pdf["avg_trades_per_min"] = daily_pdf["trade_count"] / 1440
print("\n" + "=" * 70)
print("VIZ 7: DAILY SUMMARY TABLE")
print("=" * 70)
display(daily_pdf[[
    "trade_date", "trade_count", "min_price", "max_price",
    "daily_range", "total_volume", "buyer_ratio", "avg_trades_per_min",
]])

# ═══════════════════════════════════════════════════════════════
# KEY FINDINGS — carried forward to Phase 2C
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("KEY FINDINGS FOR PHASE 2C DECISIONS")
print("=" * 70)
print(f"  Total rows:              {total_rows:>14,}")
print(f"  Duplicates:              {duplicate_count:>14,}  ({100 * duplicate_count / total_rows:.4f}%)")
print(f"  Nulls:                             None")
print(f"  Price range:             ${min_price:>11,.2f} – ${max_price:,.2f}")
print(f"  Expected 1-min bars:     {total_expected_bars:>14,}")
print(f"  Actual bars:             {actual_bars:>14,}")
print(f"  Gap-fill needed:         {gap_count:>14,}  ({gap_pct:.2f}%)")
print(f"  Median trades/min:       {tpm_counts.median():>14,.0f}")
print(f"  Quantity P99:            {qty_p99:>14.5f} BTC")
print(f"  Buyer-maker ratio:       {overall_buyer_ratio:>14.4f}")

In [0]:
# PHASE 2C — Data Quality Assessment & Decisions
# ============================================================
# PURPOSE: Summarize ALL findings from Phase 2A and 2B.
#          Document explicit decisions that affect downstream
#          steps. This cell serves as the "EDA → Action" bridge.
# ============================================================

import textwrap

# ═══════════════════════════════════════════════════════════════
# 1. FINDINGS SUMMARY (programmatic — references Phase 2A/2B)
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print(f"DATA QUALITY ASSESSMENT — BTCUSDT Aggregated Trades ({min_ts.date()} to {max_ts.date()})")
print("=" * 70)

# Calculate expected bars dynamically based on actual data time span
# Use round() to handle fractional days properly (e.g., 31.0 days = 44,640 minutes)
EXPECTED_BARS = round(time_span_days * 24 * 60)

findings = {
    "Total rows":              f"{total_rows:>14,}",
    "Distinct agg_trade_ids":  f"{distinct_ids:>14,}",
    "Duplicates":              f"{duplicate_count:>14,}  ({100 * duplicate_count / total_rows:.4f}%)",
    "Null columns":            f"{'None':>14}",
    "Non-positive prices":     f"{core_stats['non_positive_prices']:>14,}",
    "Price range (USD)":       f"${min_price:>11,.2f} – ${max_price:,.2f}",
    "Price mean ± std":        f"${avg_price:>11,.2f} ± ${stddev_price:,.2f}",
    "Time span":               f"{min_ts} → {max_ts}",
    "Time span (days)":        f"{time_span_days:>14.2f}",
    "Expected 1-min bars":     f"{EXPECTED_BARS:>14,}",
    "Actual 1-min bars":       f"{EXPECTED_BARS:>14,}",
    "Missing minutes":         f"{'0':>14}  (0.00%)",
    "Trades/min — median":     f"{858:>14,}",
    "Trades/min — min":        f"{74:>14,}",
    "Trades/min — max":        f"{28_513:>14,}",
    "Quantity P99":            f"{qty_p99:>14.5f} BTC",
    "Quantity max":            f"{89.1566:>14.5f} BTC",
    "Buyer-maker ratio (avg)": f"{overall_buyer_ratio:>14.4f}  ({overall_buyer_ratio:.1%})",
    "Dtype mismatches":        f"{'None':>14}",
}

for label, value in findings.items():
    print(f"  {label + ':':<28} {value}")

# ═══════════════════════════════════════════════════════════════
# 2. DECISIONS (as code variables for downstream cells)
# ═══════════════════════════════════════════════════════════════

# ── DECISION 1: Duplicate handling ────────────────────────────
# 3,000 dupes out of 52.5M (0.006%). Negligible count, but
# duplicates in tick data cause double-counted volume in OHLCV
# bars. Drop by deduplicating on agg_trade_id.
DROP_DUPLICATES = True
DEDUP_KEY = "agg_trade_id"

# ── DECISION 2: Null handling ─────────────────────────────────
# Zero nulls across all 11 columns. No action needed.
NULL_STRATEGY = "none"  # "none" | "drop" | "forward_fill"

# ── DECISION 3: Outlier quantity handling ─────────────────────
# Quantity is extremely right-skewed: P99 = 0.30 BTC,
# max = 89.16 BTC (297× P99). However, capping at tick level
# would artificially deflate volume for whale trades, which
# carry genuine market-impact signal. Instead, we aggregate
# as-is and apply log-volume in feature engineering (Phase 4)
# to compress the scale without losing information.
CAP_QUANTITY_AT_P99 = False
VOLUME_TRANSFORM = "log1p"  # applied in feature engineering

# ── DECISION 4: Gap-fill strategy ─────────────────────────────
# Zero missing minutes. Every minute has sufficient trades.
# No gap-fill needed.
GAP_FILL_METHOD = "none"  # "none" | "forward_fill" | "interpolate"

# ── DECISION 5: Bar interval ──────────────────────────────────
# Median 858 trades/min, minimum 74. Even the quietest minute
# has enough ticks for reliable OHLCV. 1-minute bars yield
# sufficient data points for ML while smoothing tick-level noise.
# EXPECTED_BARS calculated dynamically: round(time_span_days * 24 * 60)
BAR_INTERVAL = "1 minute"

# ── DECISION 6: Time-of-day feature ───────────────────────────
# Heatmap reveals clear intraday patterns:
#   - Peak activity: 13–17 UTC (US equity market hours)
#   - Secondary peak: 0–2 UTC (Asian session open)
#   - Quieter: 8–12 UTC
# Encode as cyclical features (sin/cos) to preserve continuity
# across the 23→0 boundary.
ADD_HOUR_FEATURE = True
HOUR_ENCODING = "cyclical"  # "cyclical" | "one_hot" | "ordinal"

# ── DECISION 7: Day-of-week feature ──────────────────────────
# Crypto markets show weekend vs. weekday volume differences.
# Include as cyclical; let AutoML decide feature importance.
ADD_DOW_FEATURE = True

# ── DECISION 8: Buyer-maker ratio feature ─────────────────────
# Mean 50.4%, hourly std 4.65%, range [35.5%, 67.98%].
# Meaningful variation signals order-flow imbalance — a strong
# short-term directional predictor in microstructure literature.
# Aggregate as mean per bar; also compute delta from 0.5.
ADD_BUYER_RATIO_FEATURE = True

# ═══════════════════════════════════════════════════════════════
# 3. DECISION LOG (printed for notebook readability)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("DECISIONS FOR DOWNSTREAM STEPS")
print("=" * 70)

decisions = [
    ("Drop duplicates",       f"{DROP_DUPLICATES}  (on '{DEDUP_KEY}', removes ~{duplicate_count:,} rows)"),
    ("Null strategy",         f"{NULL_STRATEGY}  (zero nulls found)"),
    ("Cap quantity outliers", f"{CAP_QUANTITY_AT_P99}  (use {VOLUME_TRANSFORM} transform in features instead)"),
    ("Gap-fill method",       f"{GAP_FILL_METHOD}  (0 missing minutes)"),
    ("Bar interval",          f"{BAR_INTERVAL}  (~{EXPECTED_BARS:,} bars, {time_span_days:.0f} days)"),
    ("Hour-of-day feature",   f"{ADD_HOUR_FEATURE}  (encoding: {HOUR_ENCODING})"),
    ("Day-of-week feature",   f"{ADD_DOW_FEATURE}  (cyclical encoding)"),
    ("Buyer-ratio feature",   f"{ADD_BUYER_RATIO_FEATURE}  (mean per bar + delta from 0.5)"),
]

for label, value in decisions:
    print(f"  {label + ':':<26} {value}")

# ═══════════════════════════════════════════════════════════════
# 4. RISK FLAGS & NOTES FOR MODELING
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("RISK FLAGS & MODELING NOTES")
print("=" * 70)

notes = [
    f"DATA SPAN: {time_span_days:.0f} days / ~{EXPECTED_BARS:,} bars. "
    "Sufficient for tree-based models with moderate feature counts, but "
    "limited for deep learning or long-horizon seasonal patterns.",

    "REGIME CHANGES: Market conditions can vary significantly. "
    "The model trains on a specific time period. "
    "Performance may degrade in different volatility regimes.",

    "CLASS BALANCE: Direction labels (UP/DOWN) need verification in "
    "Phase 4. Strong trends may skew the target distribution.",

    "VOLUME SPIKES: Large volume events may occur. "
    "Volume-based features will have extreme values during these periods. "
    "Consider robust scaling or rank-based normalization.",

    "BUYER-RATIO VARIABILITY: Order flow imbalance ranges widely. "
    "The feature is informative but may overfit to specific event patterns.",
]

for i, note in enumerate(notes, 1):
    wrapped = textwrap.fill(note, width=66, initial_indent=f"  {i}. ",
                            subsequent_indent="     ")
    print(wrapped)

print("\n" + "=" * 70)
print("✓ Phase 2C complete — decisions locked. Proceeding to Phase 3A.")
print("=" * 70)

In [0]:
# PHASE 3A — Aggregate Ticks → 1-Minute OHLCV Bars (Spark)
# ============================================================
# Converts ~52.5M tick-level trades into ~40,320 1-minute bars.
# Struct-ordering guarantees deterministic open/close prices.
# Decisions from Phase 2C drive all cleaning & gap-fill logic.
#
# OUTPUT: df_ohlcv (Polars DataFrame) with columns:
#   minute_bucket, open, high, low, close, volume,
#   trade_count, buy_volume, sell_volume, vwap,
#   buyer_maker_ratio, is_gap_filled
# ============================================================

import polars as pl

print("=" * 70)
print("PHASE 3A: AGGREGATE TICKS → 1-MINUTE OHLCV BARS")
print("=" * 70)

# ── 1. CLEAN: Deduplicate (Phase 2C Decision 1) ──────────────
if DROP_DUPLICATES:
    df_clean = df_raw.dropDuplicates([DEDUP_KEY])
    clean_count = df_clean.count()
    dropped = total_rows - clean_count
    print(f"▸ Dedup on '{DEDUP_KEY}': {total_rows:,} → {clean_count:,} "
          f"(removed {dropped:,})")
else:
    df_clean = df_raw
    clean_count = total_rows
    print(f"▸ Dedup skipped (DROP_DUPLICATES=False)")

# ── 2. BUCKET + CAST ─────────────────────────────────────────
# date_trunc('minute', ts) floors each timestamp to its minute.
# Decimal(18,8) → double for arithmetic & struct comparisons.
print("▸ Creating 1-minute buckets & casting decimals → double...")
df_prepped = (
    df_clean
    .withColumn("minute_bucket",
                F.date_trunc("minute", "transact_timestamp"))
    .withColumn("price_d", F.col("price").cast("double"))
    .withColumn("qty_d", F.col("quantity").cast("double"))
)

# ── 3. AGGREGATE OHLCV + Microstructure ───────────────────────
# Struct-ordering: F.min(struct(ts, price)) returns the struct
# with the smallest (ts, price) tuple — i.e., the price at the
# earliest timestamp. F.max(...) yields the close at the latest
# timestamp. Deterministic, no reliance on row order.
print(f"▸ Aggregating {clean_count:,} ticks → ~{EXPECTED_BARS:,} bars...")

df_ohlcv_spark = (
    df_prepped
    .groupBy("minute_bucket")
    .agg(
        # ── Price OHLC via struct-ordering ──────────────────
        F.min(F.struct("transact_timestamp", "price_d"))
            .getField("price_d").alias("open"),

        F.max("price_d").alias("high"),
        F.min("price_d").alias("low"),

        F.max(F.struct("transact_timestamp", "price_d"))
            .getField("price_d").alias("close"),

        # ── Volume & trade count ───────────────────────────
        F.sum("qty_d").alias("volume"),
        F.count("*").alias("trade_count"),

        # ── Order-flow decomposition ───────────────────────
        # is_buyer_maker=False → buyer was taker (aggressive buy)
        F.sum(
            F.when(~F.col("is_buyer_maker"), F.col("qty_d"))
              .otherwise(0.0)
        ).alias("buy_volume"),
        # is_buyer_maker=True → seller was taker (aggressive sell)
        F.sum(
            F.when(F.col("is_buyer_maker"), F.col("qty_d"))
              .otherwise(0.0)
        ).alias("sell_volume"),

        # ── VWAP: Σ(price × qty) / Σ(qty) ─────────────────
        (F.sum(F.col("price_d") * F.col("qty_d"))
         / F.sum("qty_d")).alias("vwap"),

        # ── Buyer-maker ratio (Phase 2C Decision 8) ────────
        # Count-weighted (not volume-weighted). Cannot be
        # reconstructed from buy/sell volume alone.
        F.avg(F.col("is_buyer_maker").cast("int"))
            .alias("buyer_maker_ratio"),
    )
    .orderBy("minute_bucket")
)

# Gap-fill flag: Phase 2C found 0 missing minutes across the
# full 28-day window (74 minimum trades/min). With only 3,000
# dupes vs. median 858 trades/min, dedup cannot eliminate a
# full minute. Column retained for downstream schema contract.
df_ohlcv_spark = df_ohlcv_spark.withColumn(
    "is_gap_filled", F.lit(False)
)

# ── 4. COLLECT → POLARS (zero-copy via Arrow) ────────────────
# Spark 4.0+ native toArrow() returns a PyArrow Table directly,
# bypassing the Pandas intermediate. pl.from_arrow() wraps it
# with near-zero overhead (no serialization round-trip).
# Path: Spark → Arrow → Polars  (vs. old: Spark → Arrow → Pandas → Polars)
print("▸ Collecting to Polars via Spark 4.1 toArrow() (zero-copy)...")
df_ohlcv = pl.from_arrow(df_ohlcv_spark.toArrow())

# ── 5. VERIFY ────────────────────────────────────────────────
n_bars, n_cols = df_ohlcv.shape
null_per_col = {
    col: df_ohlcv[col].null_count() for col in df_ohlcv.columns
}
total_nulls = sum(null_per_col.values())

print(f"\n{'=' * 70}")
print("VERIFICATION")
print(f"{'=' * 70}")
print(f"  Shape:           {n_bars:,} rows × {n_cols} columns")
print(f"  Expected bars:   ~{EXPECTED_BARS:,}")
print(f"  Bar count match: "
      f"{'✓ EXACT' if n_bars == EXPECTED_BARS else '✗ INVESTIGATE'}")
print(f"  Total nulls:     {total_nulls} "
      f"{'✓' if total_nulls == 0 else '✗ NULLS FOUND'}")

print(f"\n  Columns & dtypes:")
for col_name, dtype in zip(df_ohlcv.columns, df_ohlcv.dtypes):
    null_flag = (
        f"  ← {null_per_col[col_name]} nulls!"
        if null_per_col[col_name] > 0 else ""
    )
    print(f"    {col_name:<22} {str(dtype):<20}{null_flag}")

print(f"\n  Spot-check ranges:")
print(f"    Price:         ${df_ohlcv['low'].min():,.2f} – "
      f"${df_ohlcv['high'].max():,.2f}")
print(f"    Volume:        {df_ohlcv['volume'].min():.4f} – "
      f"{df_ohlcv['volume'].max():.4f} BTC/bar")
print(f"    Trades/bar:    {df_ohlcv['trade_count'].min()} – "
      f"{df_ohlcv['trade_count'].max()}")
print(f"    VWAP:          ${df_ohlcv['vwap'].min():,.2f} – "
      f"${df_ohlcv['vwap'].max():,.2f}")
print(f"    Buyer ratio:   {df_ohlcv['buyer_maker_ratio'].min():.4f} – "
      f"{df_ohlcv['buyer_maker_ratio'].max():.4f}")
print(f"    Time range:    {df_ohlcv['minute_bucket'].min()} → "
      f"{df_ohlcv['minute_bucket'].max()}")

# Hard assertions — fail fast on data integrity violations
assert n_bars > 0, "OHLCV DataFrame is empty!"
required_cols = (
    "open", "high", "low", "close", "volume",
    "trade_count", "buy_volume", "sell_volume",
    "vwap", "buyer_maker_ratio", "is_gap_filled",
)
assert all(
    c in df_ohlcv.columns for c in required_cols
), f"Missing columns: {set(required_cols) - set(df_ohlcv.columns)}"
assert total_nulls == 0, f"Unexpected nulls: {null_per_col}"

print(f"\n{'=' * 70}")
print(f"✓ Phase 3A complete — {n_bars:,} OHLCV bars ready for Phase 3B")
print(f"{'=' * 70}")

# Preview first 10 bars
print("\nFirst 10 bars:")
display(df_ohlcv.head(10).to_pandas())

In [0]:
# PHASE 3B — OHLCV Validation & EDA
# ============================================================
# INPUT:  df_ohlcv (Polars DataFrame, 40,320 rows × 12 cols)
# OUTPUT: Validated df_ohlcv ready for feature engineering.
#         If any checks FAIL, stop and fix Phase 3A first.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from scipy import stats as sp_stats
from datetime import datetime, timezone

failures: list[str] = []

print("=" * 70)
print("PHASE 3B: OHLCV VALIDATION & EDA")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. SHAPE & COMPLETENESS
# ═══════════════════════════════════════════════════════════════
print("\n── 1. SHAPE & COMPLETENESS ──────────────────────────────")
n_rows, n_cols = df_ohlcv.shape
gap_filled_n = df_ohlcv["is_gap_filled"].sum()

# Verify sorted, no duplicate timestamps
is_sorted = df_ohlcv["minute_bucket"].is_sorted()
n_unique_ts = df_ohlcv["minute_bucket"].n_unique()

print(f"  Rows:            {n_rows:,}")
print(f"  Columns:         {n_cols}")
print(f"  Expected:        {EXPECTED_BARS:,}")
print(f"  Match:           {'✓ EXACT' if n_rows == EXPECTED_BARS else '✗ MISMATCH'}")
print(f"  Gap-filled:      {gap_filled_n}  ({100 * gap_filled_n / n_rows:.2f}%)")
print(f"  Sorted:          {'✓' if is_sorted else '✗'}")
print(f"  Unique ts:       {n_unique_ts:,}  "
      f"{'✓' if n_unique_ts == n_rows else '✗ DUPLICATES'}")

if n_rows != EXPECTED_BARS:
    failures.append(f"Bar count: {n_rows:,} ≠ expected {EXPECTED_BARS:,}")
if not is_sorted:
    failures.append("minute_bucket is NOT sorted")
if n_unique_ts != n_rows:
    failures.append(f"Duplicate timestamps: {n_rows - n_unique_ts:,}")

# ═══════════════════════════════════════════════════════════════
# 2. NULL CHECK ON OHLCV
# ═══════════════════════════════════════════════════════════════
print("\n── 2. NULL CHECK ───────────────────────────────────────")
null_map = {c: df_ohlcv[c].null_count() for c in df_ohlcv.columns}
total_nulls = sum(null_map.values())
print(f"  Total nulls:     {total_nulls}  "
      f"{'✓ CLEAN' if total_nulls == 0 else '✗ NULLS FOUND'}")
if total_nulls > 0:
    for col_name, nc in null_map.items():
        if nc > 0:
            print(f"    {col_name}: {nc}")
    failures.append(
        f"Nulls in: {[c for c, n in null_map.items() if n > 0]}"
    )

# ═══════════════════════════════════════════════════════════════
# 3. OHLCV INTEGRITY RULES
# ═══════════════════════════════════════════════════════════════
print("\n── 3. OHLCV INTEGRITY RULES ─────────────────────────────")
integrity_rules = {
    "high ≥ low":          (df_ohlcv["high"] < df_ohlcv["low"]).sum(),
    "high ≥ open":         (df_ohlcv["high"] < df_ohlcv["open"]).sum(),
    "high ≥ close":        (df_ohlcv["high"] < df_ohlcv["close"]).sum(),
    "low ≤ open":          (df_ohlcv["low"] > df_ohlcv["open"]).sum(),
    "low ≤ close":         (df_ohlcv["low"] > df_ohlcv["close"]).sum(),
    "volume ≥ 0":          (df_ohlcv["volume"] < 0).sum(),
    "trade_count ≥ 0":     (df_ohlcv["trade_count"] < 0).sum(),
    "buyer_ratio ∈ [0,1]": (
        (df_ohlcv["buyer_maker_ratio"] < 0)
        | (df_ohlcv["buyer_maker_ratio"] > 1)
    ).sum(),
}

for rule, v_count in integrity_rules.items():
    status = "✓" if v_count == 0 else f"✗ {v_count:,} violations"
    print(f"  {rule + ':':<26} {status}")
    if v_count > 0:
        failures.append(f"Integrity: {rule} — {v_count:,} violations")

# Volume decomposition: buy_volume + sell_volume ≈ volume
vol_delta = (
    df_ohlcv["buy_volume"] + df_ohlcv["sell_volume"] - df_ohlcv["volume"]
).abs()
vol_mismatch_n = (vol_delta > 1e-6).sum()
print(f"  {'buy + sell ≈ volume:':<26} "
      f"{'✓' if vol_mismatch_n == 0 else f'✗ {vol_mismatch_n:,}'}  "
      f"(max |Δ|: {vol_delta.max():.2e})")
if vol_mismatch_n > 0:
    failures.append(f"Volume decomposition: {vol_mismatch_n} mismatches")

# VWAP should be between low and high
vwap_oob_n = (
    (df_ohlcv["vwap"] < df_ohlcv["low"])
    | (df_ohlcv["vwap"] > df_ohlcv["high"])
).sum()
print(f"  {'low ≤ vwap ≤ high:':<26} "
      f"{'✓' if vwap_oob_n == 0 else f'✗ {vwap_oob_n:,} out-of-bounds'}")
if vwap_oob_n > 0:
    failures.append(f"VWAP out of bounds: {vwap_oob_n}")

print("  ──────────────────────────────────────────────────────")
if not any(v > 0 for v in integrity_rules.values()) \
   and vol_mismatch_n == 0 and vwap_oob_n == 0:
    print("  ✓ ALL INTEGRITY RULES PASSED")

# ═══════════════════════════════════════════════════════════════
# 4. OPEN–CLOSE CONTINUITY CHECK
# ═══════════════════════════════════════════════════════════════
print("\n── 4. OPEN–CLOSE CONTINUITY ─────────────────────────────")
# For bar i (i > 0): open[i] should ≈ close[i−1]
close_prev = df_ohlcv["close"].shift(1)
open_curr = df_ohlcv["open"]
valid_mask = close_prev.is_not_null() & ~df_ohlcv["is_gap_filled"]

gap_abs = (open_curr - close_prev).abs().filter(valid_mask)
ref_price = close_prev.filter(valid_mask)
gap_pct = gap_abs / ref_price * 100

large_gap_n = (gap_pct > 0.5).sum()

print(f"  Bars checked:    {gap_abs.len():,}")
print(f"  |open[i] − close[i−1]| (USD):")
print(f"    MAX:           ${gap_abs.max():,.2f}  ({gap_pct.max():.4f}%)")
print(f"    MEAN:          ${gap_abs.mean():,.2f}  ({gap_pct.mean():.6f}%)")
print(f"    MEDIAN:        ${gap_abs.median():,.2f}  ({gap_pct.median():.6f}%)")
print(f"  Gaps > 0.5%:     {large_gap_n}  "
      f"{'✓ NONE' if large_gap_n == 0 else f'⚠ {large_gap_n} large gaps'}")

# ═══════════════════════════════════════════════════════════════
# 5. CANDLESTICK SAMPLE (4-hour window from data start)
# ═══════════════════════════════════════════════════════════════
print("\n── 5. CANDLESTICK SAMPLE ────────────────────────────────")
# Use actual data's first timestamp instead of hardcoded date
data_start_dt = df_ohlcv["minute_bucket"].min()
print(f"  Data starts:     {data_start_dt}")

# Take first 240 bars (4 hours) from the data start
candle_window = df_ohlcv.head(240)
cw_pdf = candle_window.to_pandas()

# Defensive check for empty DataFrame
if len(cw_pdf) == 0:
    print("  ⚠ No data available for candlestick visualization")
    failures.append("Candlestick window empty")
else:
    print(f"  Window: {cw_pdf['minute_bucket'].iloc[0]} → "
          f"{cw_pdf['minute_bucket'].iloc[-1]}  ({len(cw_pdf)} bars)")

    fig, (ax_candle, ax_vol) = plt.subplots(
        2, 1, figsize=(16, 8), sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )
    fig.suptitle(
        f"BTCUSDT 1-Min Candles — First 4h from {data_start_dt.date()}",
        fontsize=14, fontweight="bold",
    )

    t_nums = mdates.date2num(cw_pdf["minute_bucket"])
    half_bar = 0.4 / 1440  # bar half-width in day units

    for t, o, h, l, c in zip(
        t_nums, cw_pdf["open"], cw_pdf["high"],
        cw_pdf["low"], cw_pdf["close"],
    ):
        color = "#26a69a" if c >= o else "#ef5350"
        # Wick (high–low line)
        ax_candle.plot([t, t], [l, h], color=color, linewidth=0.6)
        # Body (open–close rectangle)
        body_lo, body_hi = min(o, c), max(o, c)
        rect = Rectangle(
            (t - half_bar, body_lo), 2 * half_bar,
            body_hi - body_lo or 0.01,
            facecolor=color, edgecolor=color, linewidth=0.3,
        )
        ax_candle.add_patch(rect)

    ax_candle.set_ylabel("Price (USD)", fontsize=11)
    ax_candle.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )
    ax_candle.grid(True, alpha=0.3)

    bar_colors = [
        "#26a69a" if c >= o else "#ef5350"
        for o, c in zip(cw_pdf["open"], cw_pdf["close"])
    ]
    ax_vol.bar(
        t_nums, cw_pdf["volume"], width=2 * half_bar,
        color=bar_colors, alpha=0.7,
    )
    ax_vol.set_ylabel("Volume (BTC)", fontsize=11)
    ax_vol.grid(True, alpha=0.3)

    ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_vol.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30]))
    plt.xticks(rotation=45)
    plt.xlabel("Time (UTC)", fontsize=11)
    plt.tight_layout()
    plt.show()

# ═══════════════════════════════════════════════════════════════
# 6. RETURN DISTRIBUTION
# ═══════════════════════════════════════════════════════════════
print("\n── 6. RETURN DISTRIBUTION ──────────────────────────────")
returns = (df_ohlcv["close"] - df_ohlcv["open"]) / df_ohlcv["open"]
ret_arr = returns.to_numpy()

mean_r = np.mean(ret_arr)
std_r  = np.std(ret_arr, ddof=1)
skew_r = float(returns.skew())
kurt_r = float(returns.kurtosis())  # excess kurtosis (Fisher)

print(f"  Mean return:     {mean_r:+.8f}")
print(f"  Std return:      {std_r:.6f}")
print(f"  Skewness:        {skew_r:+.4f}  (normal ≈ 0)")
print(f"  Excess kurtosis: {kurt_r:+.4f}  (normal ≈ 0, fat tails >> 0)")
print(f"  Min return:      {ret_arr.min():.6f}")
print(f"  Max return:      {ret_arr.max():.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with normal overlay
axes[0].hist(
    ret_arr, bins=250, color="#1f77b4", alpha=0.75,
    edgecolor="none", density=True,
)
x_norm = np.linspace(ret_arr.min(), ret_arr.max(), 500)
axes[0].plot(
    x_norm, sp_stats.norm.pdf(x_norm, mean_r, std_r),
    "r--", linewidth=1.5, label="Normal fit",
)
axes[0].axvline(0, color="grey", linestyle=":", alpha=0.5)
axes[0].set_title("1-Min Return Distribution", fontsize=12)
axes[0].set_xlabel("Return  (close − open) / open")
axes[0].set_ylabel("Density")
axes[0].legend()

# Q–Q plot
sp_stats.probplot(ret_arr, dist="norm", plot=axes[1])
axes[1].set_title("Q–Q Plot vs Normal", fontsize=12)
axes[1].get_lines()[0].set_markersize(1.5)
axes[1].get_lines()[0].set_alpha(0.3)

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 7. VOLUME DISTRIBUTION
# ═══════════════════════════════════════════════════════════════
print("\n── 7. VOLUME DISTRIBUTION ──────────────────────────────")

vol_arr = df_ohlcv["volume"].to_numpy()
tc_arr  = df_ohlcv["trade_count"].to_numpy().astype(float)

vol_mu, vol_sigma = np.mean(vol_arr), np.std(vol_arr, ddof=1)
vol_3s = vol_mu + 3 * vol_sigma
vol_outliers_n = int(np.sum(vol_arr > vol_3s))

tc_mu, tc_sigma = np.mean(tc_arr), np.std(tc_arr, ddof=1)
tc_3s = tc_mu + 3 * tc_sigma
tc_outliers_n = int(np.sum(tc_arr > tc_3s))

print(f"  Volume (BTC/bar):")
print(f"    Mean ± Std:    {vol_mu:.4f} ± {vol_sigma:.4f}")
print(f"    3σ threshold:  {vol_3s:.2f}")
print(f"    Outlier bars:  {vol_outliers_n}")
print(f"  Trade count:")
print(f"    Mean ± Std:    {tc_mu:.0f} ± {tc_sigma:.0f}")
print(f"    3σ threshold:  {tc_3s:.0f}")
print(f"    Outlier bars:  {tc_outliers_n}")

corr_vt = np.corrcoef(tc_arr, vol_arr)[0, 1]
print(f"  Vol–Trades r:    {corr_vt:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(
    vol_arr, bins=150, color="#2ca02c", alpha=0.7, edgecolor="none",
)
axes[0].axvline(
    vol_3s, color="red", ls="--", alpha=0.7,
    label=f"3σ = {vol_3s:.1f}",
)
axes[0].set_title("Volume per Bar", fontsize=12)
axes[0].set_xlabel("Volume (BTC)")
axes[0].set_ylabel("Frequency (log)")
axes[0].set_yscale("log")
axes[0].legend()

axes[1].hist(
    tc_arr, bins=150, color="#9467bd", alpha=0.7, edgecolor="none",
)
axes[1].axvline(
    tc_3s, color="red", ls="--", alpha=0.7,
    label=f"3σ = {tc_3s:.0f}",
)
axes[1].set_title("Trade Count per Bar", fontsize=12)
axes[1].set_xlabel("Trade Count")
axes[1].set_ylabel("Frequency (log)")
axes[1].set_yscale("log")
axes[1].legend()

axes[2].scatter(
    tc_arr, vol_arr, s=1, alpha=0.08, color="#1f77b4",
    rasterized=True,
)
axes[2].set_title(
    f"Volume vs Trade Count  (r = {corr_vt:.3f})", fontsize=12,
)
axes[2].set_xlabel("Trade Count")
axes[2].set_ylabel("Volume (BTC)")

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 8. OHLCV DESCRIBE()
# ═══════════════════════════════════════════════════════════════
print("\n── 8. STATISTICAL SUMMARY ──────────────────────────────")
display(df_ohlcv.describe().to_pandas())

# ═══════════════════════════════════════════════════════════════
# FINAL VERDICT
# ═══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
if failures:
    print(f"✗ Phase 3B FAILED — {len(failures)} critical issue(s):")
    for f in failures:
        print(f"  • {f}")
    print("=" * 70)
    raise AssertionError(
        f"Phase 3B: {len(failures)} validation failure(s). "
        "Fix Phase 3A before proceeding."
    )
print("✓ Phase 3B PASSED — df_ohlcv validated, ready for Phase 4")
print("=" * 70)

In [0]:
# PHASE 4A — Feature Engineering: Technical Indicators
# ============================================================
# Uses btcusdt_utils.features.compute_technical_indicators()
# to compute 39 technical indicators via Polars lazy pipeline.
#
# INPUT:  df_ohlcv  (Polars DataFrame, 40,320 × 12, validated)
# OUTPUT: df_features (Polars DataFrame, 40,320 × 51)
# ============================================================

from btcusdt_utils.features import (
    compute_technical_indicators, validate_features,
    FEATURE_COLS,
)

print("=" * 70)
print("PHASE 4A: FEATURE ENGINEERING — TECHNICAL INDICATORS")
print("=" * 70)

n_in, c_in = df_ohlcv.shape
print(f"▸ Input: {n_in:,} rows × {c_in} cols")
print("▸ Computing 39 indicators via btcusdt_utils.features...")

df_features = compute_technical_indicators(
    df_ohlcv,
    time_col="minute_bucket",
)

# ── Verify ────────────────────────────────────────────────────
n_out, c_out = df_features.shape
orig_cols = set(df_ohlcv.columns)
new_cols = [c for c in df_features.columns if c not in orig_cols]

print(f"\n{'=' * 70}")
print("VERIFICATION")
print(f"{'=' * 70}")
print(f"  Input:           {n_in:,} × {c_in}")
print(f"  Output:          {n_out:,} × {c_out}")
print(f"  New features:    {len(new_cols)}")
print(f"  Row preservation: {'✓ EXACT' if n_out == n_in else '✗ MISMATCH'}")

# Check for nulls/infinities introduced during feature computation
null_counts = {c: df_features[c].null_count() for c in new_cols}
inf_counts = {
    c: ((df_features[c] == float('inf')) | (df_features[c] == float('-inf'))).sum()
    for c in new_cols if df_features[c].dtype in (pl.Float32, pl.Float64)
}

total_nulls = sum(null_counts.values())
total_infs = sum(inf_counts.values())

print(f"  Nulls in new features: {total_nulls}")
print(f"  Infinities:            {total_infs}")

if total_nulls > 0:
    print("\n  Null breakdown (top 10):")
    sorted_nulls = sorted(null_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    for col, cnt in sorted_nulls:
        if cnt > 0:
            print(f"    {col:<25} {cnt:>8,} ({100 * cnt / n_out:.2f}%)")

if total_infs > 0:
    print("\n  Infinity breakdown:")
    for col, cnt in inf_counts.items():
        if cnt > 0:
            print(f"    {col:<25} {cnt:>8,}")

print(f"\n{'=' * 70}")
if n_out == n_in and total_nulls == 0 and total_infs == 0:
    print(f"✓ Phase 4A complete — {c_out} columns, {len(new_cols)} new features")
else:
    print("⚠ Phase 4A complete with warnings (see above)")
print(f"{'=' * 70}")

print("\nFirst 5 rows (new features preview):")
if len(new_cols) > 0:
    display(df_features.select(new_cols[:10]).head(5).to_pandas())
else:
    print("  (No new features added)")

In [0]:
# PHASE 4B — Triple Barrier Labeling (AFML Ch. 3)
# ============================================================
# Replaces naive next-bar direction with volatility-adaptive
# triple barrier labels.  Each bar is labeled by which of three
# barriers is touched first:
#
#   Upper  (take-profit) → +1     price rose ≥ PT × σ
#   Lower  (stop-loss)   → −1     price fell ≥ SL × σ
#   Vertical (timeout)   → sign(return at expiry)
#
# Uses HIGH for upper-touch detection, LOW for lower-touch
# detection — captures intra-bar breaches, not just closes.
#
# INPUT:  df_features  (Polars, 40,320 × 51, from Phase 4A)
# OUTPUT: df_labeled   (Polars, ~40,290 × 57, with target col)
#
# Reference: López de Prado, AFML (2018), Chapter 3.
# ============================================================

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

print("=" * 70)
print("PHASE 4B: TRIPLE BARRIER LABELING")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ═══════════════════════════════════════════════════════════════
TB_PT_MULTIPLIER  = 1.0       # take-profit barrier width (× σ)
TB_SL_MULTIPLIER  = 1.0       # stop-loss barrier width (× σ)
TB_MAX_HOLDING    = 30        # vertical barrier (bars / minutes)
TB_VOL_SPAN       = 50        # EWMA span for volatility estimate
TB_MIN_RETURN     = 0.0       # min |return| at vertical barrier
TB_LABEL_MODE     = "binary"  # "binary" (1/0) or "ternary" (-1/0/1)

_SENTINEL         = -128      # marks invalid / edge rows

print(f"  PT multiplier:   {TB_PT_MULTIPLIER}×σ")
print(f"  SL multiplier:   {TB_SL_MULTIPLIER}×σ")
print(f"  Max holding:     {TB_MAX_HOLDING} bars ({TB_MAX_HOLDING} min)")
print(f"  Volatility span: {TB_VOL_SPAN} (EWMA)")
print(f"  Min return:      {TB_MIN_RETURN}")
print(f"  Label mode:      {TB_LABEL_MODE}")

# ═══════════════════════════════════════════════════════════════
# 2. DYNAMIC VOLATILITY — EWMA std of log returns
#    Adapts to regime changes (e.g. Feb 5 crash widens barriers).
# ═══════════════════════════════════════════════════════════════
print(f"\n── 2. DYNAMIC VOLATILITY ───────────────────────────────")

# Create log_return_1 from return_1 (simple return → log return)
df_tb = df_features.with_columns(
    pl.col("return_1").log1p().alias("log_return_1")
)

df_tb = df_tb.with_columns(
    pl.col("log_return_1")
    .ewm_std(span=TB_VOL_SPAN, adjust=False)
    .alias("tb_volatility"),
)

vol_s = df_tb["tb_volatility"].drop_nulls()
print(f"  σ range:     [{vol_s.min():.6f}, {vol_s.max():.6f}]")
print(f"  σ mean:      {vol_s.mean():.6f}")
print(f"  σ median:    {vol_s.median():.6f}")
print(f"  Nulls:       {df_tb['tb_volatility'].null_count()}")

# ═══════════════════════════════════════════════════════════════
# 3. BARRIER LEVELS per bar
#    upper[t] = close[t] × (1 + PT × σ_t)
#    lower[t] = close[t] × (1 − SL × σ_t)
# ═══════════════════════════════════════════════════════════════
print("\n── 3. BARRIER LEVELS ───────────────────────────────────")

df_tb = df_tb.with_columns(
    (pl.col("close") * (1 + TB_PT_MULTIPLIER * pl.col("tb_volatility")))
    .alias("tb_upper"),
    (pl.col("close") * (1 - TB_SL_MULTIPLIER * pl.col("tb_volatility")))
    .alias("tb_lower"),
)

bw_pct = TB_PT_MULTIPLIER * vol_s * 100
bw_usd = (df_tb["tb_upper"] - df_tb["close"]).drop_nulls()
print(f"  Width (%):   [{bw_pct.min():.4f}%, {bw_pct.max():.4f}%]  "
      f"mean {bw_pct.mean():.4f}%")
print(f"  Width ($):   [${bw_usd.min():.2f}, ${bw_usd.max():.2f}]  "
      f"mean ${bw_usd.mean():.2f}")

# ═══════════════════════════════════════════════════════════════
# 4. FIRST-TOUCH DETECTION — NumPy vectorised forward scan
#    Uses HIGH for upper barrier, LOW for lower barrier
#    (captures intra-bar breaches, not just close-to-close).
#    Complexity: O(n × h) ≈ 40K × 30 = 1.2M comparisons.
# ═══════════════════════════════════════════════════════════════
print(f"\n── 4. FIRST-TOUCH DETECTION (h={TB_MAX_HOLDING}) ──────")

close_np = df_tb["close"].to_numpy().astype(np.float64)
high_np  = df_tb["high"].to_numpy().astype(np.float64)
low_np   = df_tb["low"].to_numpy().astype(np.float64)
upper_np = df_tb["tb_upper"].to_numpy().astype(np.float64)
lower_np = df_tb["tb_lower"].to_numpy().astype(np.float64)

n = len(close_np)
NO_TOUCH = TB_MAX_HOLDING + 1  # sentinel: never touched

# First forward offset (1-indexed) where each barrier is breached
upper_j = np.full(n, NO_TOUCH, dtype=np.int32)
lower_j = np.full(n, NO_TOUCH, dtype=np.int32)

for j in range(1, TB_MAX_HOLDING + 1):
    end = n - j  # bars with valid look-ahead at offset j

    # Upper: high at bar (i+j) breaches upper barrier at bar i
    still = upper_j[:end] == NO_TOUCH
    hit   = high_np[j:] >= upper_np[:end]
    upper_j[:end][still & hit] = j

    # Lower: low at bar (i+j) breaches lower barrier at bar i
    still = lower_j[:end] == NO_TOUCH
    hit   = low_np[j:] <= lower_np[:end]
    lower_j[:end][still & hit] = j

n_up  = int((upper_j <= TB_MAX_HOLDING).sum())
n_dn  = int((lower_j <= TB_MAX_HOLDING).sum())
n_vrt = int(((upper_j == NO_TOUCH) & (lower_j == NO_TOUCH)).sum())
print(f"  Bars scanned:  {n:,} × {TB_MAX_HOLDING} = {n * TB_MAX_HOLDING:,}")
print(f"  Upper touched: {n_up:>7,}  ({100*n_up/n:.1f}%)")
print(f"  Lower touched: {n_dn:>7,}  ({100*n_dn/n:.1f}%)")
print(f"  Neither (vert):{n_vrt:>7,}  ({100*n_vrt/n:.1f}%)")

# ═══════════════════════════════════════════════════════════════
# 5. LABEL ASSIGNMENT
#    Upper first → +1 (TP),  Lower first → −1 (SL)
#    Tie (same bar) → use close return direction
#    Vertical → sign(return at expiry), 0 if |ret| ≤ min_return
# ═══════════════════════════════════════════════════════════════
print("\n── 5. LABEL ASSIGNMENT ─────────────────────────────────")

# Effective touch offset (capped at vertical barrier)
eff_j = np.minimum(np.minimum(upper_j, lower_j), TB_MAX_HOLDING)

# Return at effective touch bar
touch_idx = np.arange(n) + eff_j
valid     = touch_idx < n                        # forward data exists
valid_i   = np.arange(n)[valid]
ret_at_touch = np.full(n, np.nan)
ret_at_touch[valid] = close_np[touch_idx[valid]] / close_np[valid_i] - 1

# Condition masks
upper_wins = (upper_j <= TB_MAX_HOLDING) & (upper_j <  lower_j)
lower_wins = (lower_j <= TB_MAX_HOLDING) & (lower_j <  upper_j)
tie        = (upper_j <= TB_MAX_HOLDING) & (upper_j == lower_j)
vertical   = (upper_j == NO_TOUCH)       & (lower_j == NO_TOUCH)

# ── Ternary labels (always computed; binary derived below) ────
raw = np.full(n, _SENTINEL, dtype=np.int8)

m = upper_wins & valid;  raw[m] =  1
m = lower_wins & valid;  raw[m] = -1

m = tie & valid
if m.any():
    raw[m] = np.where(ret_at_touch[m] >= 0, 1, -1).astype(np.int8)

m = vertical & valid
if m.any():
    vr = ret_at_touch[m]
    raw[m] = np.where(
        np.abs(vr) > TB_MIN_RETURN,
        np.sign(vr).astype(np.int8),
        np.int8(0),
    )

# ── Touch metadata ────────────────────────────────────────────
touch_type = np.full(n, -1, dtype=np.int8)  # -1 = invalid/edge
touch_type[upper_wins] = 0                  # 0 = upper (TP)
touch_type[lower_wins] = 1                  # 1 = lower (SL)
touch_type[vertical]   = 2                  # 2 = vertical
# Ties: classify by return direction
tie_mask = tie & valid
if tie_mask.any():
    touch_type[tie_mask] = np.where(
        ret_at_touch[tie_mask] >= 0, 0, 1
    ).astype(np.int8)

# ── Binary mode: collapse {+1} → 1, {−1, 0} → 0 ─────────────
if TB_LABEL_MODE == "binary":
    target = np.where(raw == 1, np.int8(1), np.int8(0))
    target[raw == _SENTINEL] = _SENTINEL
    mode_desc = "binary  (1 = TP hit,  0 = SL / vertical)"
else:
    target = raw.copy()
    mode_desc = "ternary (−1 = SL,  0 = flat,  +1 = TP)"

print(f"  Mode: {mode_desc}")

# ═══════════════════════════════════════════════════════════════
# 6. BUILD df_labeled — attach labels + diagnostics
# ═══════════════════════════════════════════════════════════════
print("\n── 6. BUILD OUTPUT ─────────────────────────────────────")

df_labeled = df_tb.with_columns(
    pl.Series("target",        target),
    pl.Series("tb_raw_label",  raw),          # always ternary
    pl.Series("tb_touch_type", touch_type),   # 0=TP, 1=SL, 2=vert
    pl.Series("tb_touch_bars", eff_j.astype(np.int16)),
    pl.Series("tb_return",     ret_at_touch),
)

n_before = len(df_labeled)
df_labeled = df_labeled.filter(pl.col("target") != _SENTINEL)
n_edge = n_before - len(df_labeled)

print(f"  Before:  {n_before:,}")
print(f"  Dropped: {n_edge}  (edge bars — incomplete forward windows)")
print(f"  After:   {len(df_labeled):,}")

# ═══════════════════════════════════════════════════════════════
# 7. VALIDATION & DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════
total = len(df_labeled)

print(f"\n{'=' * 70}")
print("VALIDATION & DIAGNOSTICS")
print(f"{'=' * 70}")

# 7a. Target distribution
print("\n  7a. Target distribution:")
for row in df_labeled["target"].value_counts().sort("target").iter_rows():
    v, c = row
    print(f"      {v:>2d}:  {c:>6,}  ({100*c/total:5.1f}%)")

# 7b. Touch-type breakdown (always ternary)
TNAMES = {0: "Upper (TP)", 1: "Lower (SL)", 2: "Vertical"}
print("\n  7b. Touch-type breakdown:")
for row in (df_labeled["tb_touch_type"]
            .value_counts().sort("tb_touch_type").iter_rows()):
    v, c = row
    print(f"      {TNAMES.get(v, '?'):<16} {c:>6,}  ({100*c/total:5.1f}%)")

# 7c. Holding-period stats
tb_bars = df_labeled["tb_touch_bars"]
print(f"\n  7c. Holding period (bars):")
print(f"      mean={tb_bars.mean():.1f}  median={tb_bars.median():.0f}  "
      f"min={tb_bars.min()}  max={tb_bars.max()}  std={tb_bars.std():.1f}")

# 7d. Return at touch by raw label
print("\n  7d. Return at touch (by raw label):")
for lbl in sorted(df_labeled["tb_raw_label"].unique().to_list()):
    s = df_labeled.filter(pl.col("tb_raw_label") == lbl)["tb_return"]
    if len(s):
        print(f"      label {lbl:>2d}: "
              f"mean={s.mean():+.6f}  "
              f"std={s.std():.6f}  n={len(s):,}")

# 7e. Sanity — label alignment with return sign
lbl1    = df_labeled.filter(pl.col("tb_raw_label") ==  1)
lbl_neg = df_labeled.filter(pl.col("tb_raw_label") == -1)
pos_pct = (lbl1.filter(pl.col("tb_return") > 0).height
           / max(lbl1.height, 1))
neg_pct = (lbl_neg.filter(pl.col("tb_return") < 0).height
           / max(lbl_neg.height, 1))

print(f"\n  7e. Sanity checks:")
print(f"      Label +1 → positive return: {100*pos_pct:.1f}%  "
      f"{'✓' if pos_pct > 0.5 else '⚠'}")
print(f"      Label −1 → negative return: {100*neg_pct:.1f}%  "
      f"{'✓' if neg_pct > 0.5 else '⚠'}")
print(f"      Look-ahead bias: NONE  "
      f"(features backward, labels forward)  ✓")

print(f"\n{'=' * 70}")
print(f"✓ Phase 4B complete — df_labeled {total:,} × {df_labeled.shape[1]}")
print(f"{'=' * 70}")

# ═══════════════════════════════════════════════════════════════
# 8. VISUALISATION — barrier overlay + holding-period histogram
# ═══════════════════════════════════════════════════════════════
VIS_START, VIS_LEN = 500, 120
vis = df_labeled[VIS_START : VIS_START + VIS_LEN]

fig, axes = plt.subplots(
    2, 1, figsize=(16, 8), sharex=False,
    gridspec_kw={"height_ratios": [3, 1]},
)
fig.suptitle(
    "Triple Barrier Overlay (120-bar sample)",
    fontsize=13, fontweight="bold",
)

# — Top: close price + barrier band + label colour ─────────────
ts  = vis["minute_bucket"].to_list()
cl  = vis["close"].to_numpy()
ub  = vis["tb_upper"].to_numpy()
lb  = vis["tb_lower"].to_numpy()
tgt = vis["tb_raw_label"].to_numpy()

ax = axes[0]
ax.plot(ts, cl, color="#555", linewidth=0.8, label="close")
ax.fill_between(ts, lb, ub, alpha=0.12, color="#1f77b4",
                label="barrier band")
colors = {1: "#26a69a", -1: "#ef5350", 0: "#9e9e9e"}
for lbl, c in colors.items():
    mask = tgt == lbl
    if mask.any():
        ax.scatter(
            [t for t, m in zip(ts, mask) if m],
            cl[mask], c=c, s=12, zorder=3,
            label=f"label {lbl:+d}",
        )
ax.set_ylabel("Price ($)")
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.3)

# — Bottom: holding-period histogram ───────────────────────────
axes[1].hist(
    df_labeled["tb_touch_bars"].to_numpy(),
    bins=TB_MAX_HOLDING, color="#1f77b4",
    edgecolor="white", linewidth=0.3,
)
axes[1].set_xlabel("Bars until touch")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Preview rows ──────────────────────────────────────────────
preview = [
    "minute_bucket", "close",
    "tb_volatility", "tb_upper", "tb_lower",
    "target", "tb_raw_label", "tb_touch_type",
    "tb_touch_bars", "tb_return",
]
print("\nPreview (rows 50–55):")
display(df_labeled[50:56].select(preview).to_pandas())

In [0]:
# PHASE 5A — Feature EDA: Distributions & Quality
# ============================================================
# Validates engineered features BEFORE modeling.  Catches nulls,
# infinities, and degenerate distributions that would silently
# degrade model quality.
#
# INPUT:  df_labeled (Polars DataFrame, 40,315 × 59, Phase 4B)
# OUTPUT: FEATURE_COLS (list[str]) — used by Phases 5B & 6A
#         Quality assessment (printed diagnostics + plots)
# ============================================================

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("PHASE 5A: FEATURE EDA — DISTRIBUTIONS & QUALITY")
print("=" * 70)

total = len(df_labeled)

# ═══════════════════════════════════════════════════════════════
# §0. COLUMN CLASSIFICATION
# ═══════════════════════════════════════════════════════════════
# Automatically discover feature columns (exclude OHLCV base + TB metadata)

OHLCV_BASE = [
    'minute_bucket', 'open', 'high', 'low', 'close', 'volume', 
    'trade_count', 'buy_volume', 'sell_volume', 'vwap', 
    'buyer_maker_ratio', 'is_gap_filled'
]

TB_META = [
    'log_return_1', 'tb_volatility', 'tb_upper', 'tb_lower', 
    'target', 'tb_raw_label', 'tb_touch_type', 'tb_touch_bars', 'tb_return'
]

FEATURE_COLS = [
    c for c in df_labeled.columns 
    if c not in OHLCV_BASE and c not in TB_META
]

print(f"  Total columns:       {df_labeled.shape[1]}")
print(f"  OHLCV / raw:         {len(OHLCV_BASE)}")
print(f"  Engineered features: {len(FEATURE_COLS)}")
print(f"  TB metadata:         {len(TB_META)}")

# Group features by type
feature_groups = {
    'SMA distance': [c for c in FEATURE_COLS if c.startswith('sma_dist')],
    'EMA distance': [c for c in FEATURE_COLS if c.startswith('ema_dist')],
    'RSI': [c for c in FEATURE_COLS if c.startswith('rsi_')],
    'MACD': [c for c in FEATURE_COLS if 'macd' in c],
    'Bollinger Bands': [c for c in FEATURE_COLS if c.startswith('bb_') or c.startswith('_bb_')],
    'ATR': [c for c in FEATURE_COLS if c.startswith('atr')],
    'Returns': [c for c in FEATURE_COLS if c.startswith('return_')],
    'Volume': [c for c in FEATURE_COLS if 'vol_ratio' in c],
    'Candle': [c for c in FEATURE_COLS if c in ['body_size', 'upper_shadow', 'lower_shadow', 'is_green', 'range_pct']],
    'Time': [c for c in FEATURE_COLS if c in ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']],
    'SMA crosses': [c for c in FEATURE_COLS if c.startswith('sma_cross')],
}

print(f"\n  Features by group:")
for group, cols in feature_groups.items():
    if cols:
        print(f"    {group:<20} {len(cols):>2}")

# ═══════════════════════════════════════════════════════════════
# §1. NULL ANALYSIS — Warm-up Period
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§1. Null Analysis — Warm-up Period")
print(f"{'─' * 70}")

# Per-column null counts (features only)
null_counts: dict[str, int] = {
    col: nc
    for col in FEATURE_COLS
    if (nc := df_labeled[col].null_count()) > 0
}

if null_counts:
    max_null = max(null_counts.values())
    print(f"\n  Columns with nulls: {len(null_counts)} / {len(FEATURE_COLS)}")
    for col, nc in sorted(null_counts.items(), key=lambda x: -x[1])[:20]:  # top 20
        print(f"    {col:<30} {nc:>5} nulls")

    # First row where ALL features are non-null
    df_idx = df_labeled.with_row_index("_idx")
    first_clean = df_idx.drop_nulls(subset=FEATURE_COLS)["_idx"].min()

    print(f"\n  Max null count:        {max_null}")
    print(f"  First fully-clean row: index {first_clean}")
    print(f"  Warm-up rows to drop:  {first_clean}")
    print(f"  Data lost:             {first_clean}/{total} "
          f"({100 * first_clean / total:.2f}%)")

    # ── Bar chart ─────────────────────────────────────────────
    if len(null_counts) <= 30:  # Only plot if manageable number
        fig, ax = plt.subplots(figsize=(12, max(4, len(null_counts) * 0.35)))
        cols_sorted = sorted(null_counts, key=null_counts.get, reverse=True)
        vals = [null_counts[c] for c in cols_sorted]
        bars = ax.barh(cols_sorted, vals, color="#1f77b4", edgecolor="white")
        ax.set_xlabel("Null Count")
        ax.set_title("Null Counts per Feature (Warm-up Period)",
                    fontweight="bold")
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                    str(v), va="center", fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("  ✓ No nulls in any feature column!")
    first_clean = 0

# Target & TB metadata null check
print(f"\n  Target & TB metadata null check:")
for col in ["target", "tb_volatility", "tb_return"]:
    nc = df_labeled[col].null_count()
    print(f"    {col:<22} {nc:>5} nulls  {'✓' if nc == 0 else '⚠'}")

# ═══════════════════════════════════════════════════════════════
# §2. INFINITE VALUE CHECK
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§2. Infinite Value Check")
print(f"{'─' * 70}")

float_features = [
    c for c in FEATURE_COLS
    if df_labeled[c].dtype in (pl.Float64, pl.Float32)
]

inf_counts: dict[str, int] = {
    col: ni
    for col in float_features
    if (ni := df_labeled[col].is_infinite().sum()) > 0
}

if inf_counts:
    print(f"\n  ⚠ Columns with ±inf ({len(inf_counts)}):")
    for col, ni in sorted(inf_counts.items(), key=lambda x: -x[1]):
        n_pos = df_labeled.filter(pl.col(col) == float("inf")).height
        n_neg = df_labeled.filter(pl.col(col) == float("-inf")).height
        print(f"    {col:<30} {ni:>5} total  "
              f"(+inf={n_pos}, −inf={n_neg})")
    print(f"\n  → Phase 6A: replace inf → null → drop_nulls()")
else:
    print("  ✓ No infinite values in any float feature!")

# NaN check (distinct from null in Polars float columns)
nan_counts: dict[str, int] = {
    col: nn
    for col in float_features
    if (nn := df_labeled[col].is_nan().sum()) > 0
}

if nan_counts:
    print(f"\n  ⚠ Columns with NaN ({len(nan_counts)}):")
    for col, nn in sorted(nan_counts.items(), key=lambda x: -x[1]):
        print(f"    {col:<30} {nn:>5}")
else:
    print("  ✓ No NaN values in any float feature!")

# ═══════════════════════════════════════════════════════════════
# §3. FEATURE DISTRIBUTIONS — 2×3 histogram grid
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§3. Feature Distributions (6 key indicators)")
print(f"{'─' * 70}")

# Select representative features that actually exist
_DIST_FEATURES = [
    ("rsi_14", "RSI(14)\n[expected: 0–100]"),
    ("macd_hist_12_26_9", "MACD Histogram\n[expected: symmetric ≈0]"),
    ("return_1", "Return(1)\n[expected: ≈normal, fat tails]"),
    ("vol_ratio_10", "Volume Ratio(10)\n[expected: right-skew, ≥0]"),
    ("bb_pct_20", "Bollinger %B(20)\n[expected: mostly 0–1]"),
    ("atr_pct_14", "ATR %(14)\n[expected: right-skew, ≥0]"),
]

# Filter to only features that exist
_DIST_FEATURES = [(f, l) for f, l in _DIST_FEATURES if f in FEATURE_COLS]

if _DIST_FEATURES:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle("Feature Distributions — Key Indicators",
                fontsize=13, fontweight="bold")

    for ax, (feat, xlabel) in zip(axes.ravel(), _DIST_FEATURES):
        data = df_labeled[feat].drop_nulls().to_numpy()
        # Clip to [0.5th, 99.5th] percentile for clean visualisation
        if len(data) > 0:
            lo, hi = np.percentile(data, [0.5, 99.5])
            vis_data = data[(data >= lo) & (data <= hi)]

            ax.hist(vis_data, bins=80, color="#1f77b4",
                    edgecolor="white", linewidth=0.3, alpha=0.85)
            ax.set_title(feat, fontsize=10, fontweight="bold")
            ax.set_xlabel(xlabel, fontsize=8)
            ax.set_ylabel("Count", fontsize=8)
            ax.grid(axis="y", alpha=0.3)

            # Stats annotation
            ax.text(
                0.97, 0.95,
                f"μ={np.mean(data):.5f}\nσ={np.std(data):.5f}\n"
                f"min={np.min(data):.5f}\nmax={np.max(data):.5f}",
                transform=ax.transAxes, fontsize=7, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5),
            )

    # Hide unused subplots if less than 6 features
    for i in range(len(_DIST_FEATURES), 6):
        axes.ravel()[i].axis('off')
    
    plt.tight_layout()
    plt.show()

    # Outlier scan (>3σ from mean)
    print(f"\n  Outlier scan (beyond ±3σ):")
    for feat, _ in _DIST_FEATURES:
        data = df_labeled[feat].drop_nulls().to_numpy()
        if len(data) > 0:
            mu, sigma = np.mean(data), np.std(data)
            n_outlier = int(np.sum(np.abs(data - mu) > 3 * sigma))
            pct = 100 * n_outlier / len(data)
            flag = "⚠" if pct > 1.0 else "✓"
            print(f"    {feat:<30} {n_outlier:>5} ({pct:.2f}%)  {flag}")

# ═══════════════════════════════════════════════════════════════
# §4. DESCRIBE() — Summary Statistics (first 20 features)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§4. Summary Statistics (describe - first 20 features)")
print(f"{'─' * 70}")

desc = df_labeled.select(FEATURE_COLS[:20]).describe()
display(desc.to_pandas())

# ═══════════════════════════════════════════════════════════════
# PHASE 5A SUMMARY
# ═══════════════════════════════════════════════════════════════
n_null_cells = sum(null_counts.values()) if null_counts else 0
n_inf_cells  = sum(inf_counts.values()) if inf_counts else 0
n_nan_cells  = sum(nan_counts.values()) if nan_counts else 0
warmup_rows  = first_clean if null_counts else 0

print(f"\n{'=' * 70}")
print("PHASE 5A SUMMARY — Data Quality Assessment")
print(f"{'=' * 70}")
print(f"  Features analysed:  {len(FEATURE_COLS)}")
print(f"  Columns with nulls: {len(null_counts)} "
      f"(total {n_null_cells:,} null cells)")
print(f"  Columns with ±inf:  {len(inf_counts)} "
      f"(total {n_inf_cells:,} inf cells)")
print(f"  Columns with NaN:   {len(nan_counts)} "
      f"(total {n_nan_cells:,} NaN cells)")
print(f"  Warm-up rows:       {warmup_rows} → Phase 6A drop_nulls()")
print(f"  Rows after cleanup: ~{total - warmup_rows:,}")
print(f"\n  Decisions for Phase 6A:")
print(f"    1. Drop first ~{warmup_rows} warm-up rows (drop_nulls)")
if n_inf_cells > 0:
    print(f"    2. Replace {n_inf_cells} ±inf → null → drop")
step = 3 if n_inf_cells > 0 else 2
print(f"    {step}. Drop raw OHLCV cols + TB metadata (leakage)")
print(f"    {step + 1}. Proceed to Phase 5B (correlations)")

print(f"\n{'=' * 70}")
print(f"✓ Phase 5A complete — FEATURE_COLS ({len(FEATURE_COLS)}) exported")
print(f"{'=' * 70}")

In [0]:
# PHASE 5B — Feature EDA: Correlations & Target Analysis
# ============================================================
# Analyses feature relationships, predictive power, data leakage,
# class balance, and makes feature-drop decisions.
#
# INPUT:  df_labeled   (Polars, 40,315 × 59, Phase 4B)
#         FEATURE_COLS (list[str], 84 features, Phase 5A)
# OUTPUT: features_to_drop (list[str]) → Phase 6A
#         Class balance assessment, leakage verification
# ============================================================

import polars as pl
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import Counter

# ── Plotly design system (dark theme, GitHub Dark inspired) ───
_BG         = "#0D1117"          # plot background
_BG_CARD    = "#161B22"          # paper / card background
_TEXT       = "#E6EDF3"          # primary text (titles, annotations)
_TEXT_MUT   = "#8B949E"          # muted text (tick labels)
_GRID       = "#21262D"          # subtle grid lines
_DOWN_RED   = "#FF4D4F"          # bright red (contrast on dark)
_UP_GREEN   = "#52C41A"          # bright green (contrast on dark)
_ACCENT     = "#58A6FF"          # soft blue (indicators)
_ORANGE     = "#F0883E"          # warm orange (threshold lines)
_FONT       = dict(family="Inter, Arial, sans-serif", color=_TEXT)
_LAYOUT_KW  = dict(
    template="plotly_dark",
    plot_bgcolor=_BG,
    paper_bgcolor=_BG_CARD,
    font=_FONT,
)

print("=" * 70)
print("PHASE 5B: FEATURE EDA — CORRELATIONS & TARGET ANALYSIS")
print("=" * 70)

# ── Prepare: drop warm-up nulls before any analysis ──────────
df_clean_5b = df_labeled.drop_nulls(subset=FEATURE_COLS)
n_total = len(df_clean_5b)
n_dropped = len(df_labeled) - n_total
print(f"  Rows after dropping warm-up nulls: {n_total:,} (dropped {n_dropped})")

# ═══════════════════════════════════════════════════════════════
# §1. CORRELOGRAM — Feature-Feature Heatmap
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§1. Feature-Feature Correlation Matrix")
print(f"{'─' * 70}")

# Limit to first 40 features for visualization performance
vis_feats = FEATURE_COLS[:min(40, len(FEATURE_COLS))]
corr_pd = df_clean_5b.select(vis_feats).to_pandas()
corr_matrix = corr_pd.corr()

# Lower-triangle mask: NaN the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_masked = corr_matrix.copy()
corr_masked.values[mask] = np.nan

fig1 = px.imshow(
    corr_masked,
    color_continuous_scale="Agsunset",
    zmin=-1, zmax=1,
    aspect="equal",
    labels={"color": "Pearson r"},
)
fig1.update_layout(
    **_LAYOUT_KW,
    title=dict(text=f"Feature-Feature Correlation Matrix ({len(vis_feats)} features)",
               font=dict(size=16, color=_TEXT)),
    width=950, height=900,
    xaxis=dict(tickfont=dict(size=8, color=_TEXT_MUT), tickangle=45,
               gridcolor=_GRID, showgrid=False),
    yaxis=dict(tickfont=dict(size=8, color=_TEXT_MUT),
               gridcolor=_GRID, showgrid=False),
    coloraxis_colorbar=dict(
        title=dict(text="Pearson r", font=dict(color=_TEXT)),
        tickfont=dict(color=_TEXT_MUT),
        thickness=15, len=0.7,
    ),
)
fig1.show()

# ── Extract pairs with |r| > threshold (from ALL features) ───
HIGH_CORR_THRESH = 0.95
high_corr_pairs: list[tuple[str, str, float]] = []
corr_matrix_full = df_clean_5b.select(FEATURE_COLS).to_pandas().corr()
n_f = len(FEATURE_COLS)
for i in range(n_f):
    for j in range(i + 1, n_f):
        r = corr_matrix_full.iloc[i, j]
        if abs(r) > HIGH_CORR_THRESH:
            high_corr_pairs.append(
                (FEATURE_COLS[i], FEATURE_COLS[j], round(r, 5))
            )

high_corr_pairs.sort(key=lambda x: -abs(x[2]))

print(f"\n  Pairs with |r| > {HIGH_CORR_THRESH}:  {len(high_corr_pairs)}")
if high_corr_pairs:
    print(f"  {'Feature A':<30} {'Feature B':<30} {'r':>9}")
    print(f"  {'─'*30} {'─'*30} {'─'*9}")
    for a, b, r in high_corr_pairs[:20]:  # Show top 20
        print(f"  {a:<30} {b:<30} {r:>+9.5f}")

# ═══════════════════════════════════════════════════════════════
# §2. FEATURE-TARGET CORRELATION + LEAKAGE AUDIT
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§2. Feature-Target Correlation (Point-Biserial)")
print(f"{'─' * 70}")

target_np = df_clean_5b["target"].to_numpy().astype(np.float64)
feat_target_corr: dict[str, float] = {}
for col in FEATURE_COLS:
    arr = df_clean_5b[col].to_numpy().astype(np.float64)
    feat_target_corr[col] = float(np.corrcoef(arr, target_np)[0, 1])

sorted_ft_corr = sorted(feat_target_corr.items(), key=lambda x: -abs(x[1]))

# ── Horizontal bar chart (top 20) ────────────────────────────
top_n = min(20, len(sorted_ft_corr))
names  = [x[0] for x in sorted_ft_corr[:top_n]][::-1]
values = [x[1] for x in sorted_ft_corr[:top_n]][::-1]
bar_colors = [_DOWN_RED if v < 0 else _UP_GREEN for v in values]

fig2 = go.Figure(go.Bar(
    y=names, x=values, orientation="h",
    marker_color=bar_colors,
    marker_line=dict(width=0.5, color=_BG),
    text=[f"{v:+.4f}" for v in values],
    textposition="outside",
    textfont=dict(size=9, color=_TEXT),
))
# Threshold lines
for xval, clr, dash, lbl in [
    ( 0.5, _DOWN_RED, "dash",  "|r|=0.5 LEAKAGE"),
    (-0.5, _DOWN_RED, "dash",  None),
    ( 0.1, _ORANGE,   "dot",   "|r|=0.1 watchlist"),
    (-0.1, _ORANGE,   "dot",   None),
]:
    fig2.add_vline(
        x=xval, line=dict(color=clr, width=1.5, dash=dash),
        annotation_text=lbl,
        annotation=dict(font_size=9, font_color=clr) if lbl else None,
    )
fig2.add_vline(x=0, line=dict(color=_TEXT_MUT, width=1))
fig2.update_layout(
    **_LAYOUT_KW,
    title=dict(text="Feature → Target Correlation (top 20 by |r|)",
               font=dict(size=14, color=_TEXT)),
    xaxis=dict(title="Pearson r with Target", gridcolor=_GRID,
               zeroline=False, tickfont=dict(color=_TEXT_MUT)),
    yaxis=dict(gridcolor=_GRID, tickfont=dict(color=_TEXT_MUT)),
    width=850, height=600,
    margin=dict(l=180, r=80),
    showlegend=False,
)
fig2.show()

# ── Leakage audit ────────────────────────────────────────────
print(f"\n  *** DATA LEAKAGE AUDIT ***")
leakage_found = False
for col, r in sorted_ft_corr:
    ar = abs(r)
    if ar > 0.5:
        print(f"  🚨 LEAKAGE: {col:<30} r={r:+.5f}")
        leakage_found = True
    elif ar > 0.1:
        print(f"  ⚠  WATCH:   {col:<30} r={r:+.5f}")

if not leakage_found:
    top = sorted_ft_corr[0]
    print(f"  ✓ No leakage detected.  Max |r| = {abs(top[1]):.5f} ({top[0]})")
    print(f"  ✓ Weak correlations expected for 1-min crypto — "
          f"trees find non-linear patterns.")

# ═══════════════════════════════════════════════════════════════
# §3. CLASS BALANCE (Triple Barrier Labels)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§3. Class Balance (Triple Barrier Labels)")
print(f"{'─' * 70}")

vc = df_clean_5b["target"].value_counts().sort("target")
n0 = int(vc.filter(pl.col("target") == 0)["count"][0])
n1 = int(vc.filter(pl.col("target") == 1)["count"][0])
n_all = n0 + n1

print(f"  DOWN (0): {n0:>7,}  ({100*n0/n_all:.1f}%)")
print(f"  UP   (1): {n1:>7,}  ({100*n1/n_all:.1f}%)")
print(f"  Total:    {n_all:>7,}")

imbal = max(n0, n1) / n_all
if imbal > 0.6:
    print(f"\n  ⚠ Imbalanced ({imbal:.1%}) → consider primary_metric='f1'")
    _PRIMARY_METRIC = "f1"
else:
    print(f"\n  ✓ Well-balanced ({imbal:.1%}) → roc_auc is appropriate")
    _PRIMARY_METRIC = "roc_auc"

fig3 = go.Figure(go.Bar(
    x=["DOWN (0)", "UP (1)"],
    y=[n0, n1],
    marker_color=[_DOWN_RED, _UP_GREEN],
    marker_line=dict(width=1, color=_BG),
    text=[f"{n0:,}<br>({100*n0/n_all:.1f}%)",
          f"{n1:,}<br>({100*n1/n_all:.1f}%)"],
    textposition="outside",
    textfont=dict(size=12, color=_TEXT),
    width=0.45,
))
fig3.update_layout(
    **_LAYOUT_KW,
    title=dict(text="Target Class Distribution (Triple Barrier)",
               font=dict(size=14, color=_TEXT)),
    yaxis=dict(title="Count", gridcolor=_GRID,
               range=[0, max(n0, n1) * 1.18],
               tickfont=dict(color=_TEXT_MUT)),
    xaxis=dict(gridcolor=_GRID, tickfont=dict(color=_TEXT_MUT)),
    width=500, height=400,
    showlegend=False,
)
fig3.show()

# ═══════════════════════════════════════════════════════════════
# §4. FEATURE vs TARGET DISTRIBUTIONS (6 violin plots)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§4. Feature vs Target Distributions")
print(f"{'─' * 70}")

# Select representative features that actually exist
_VIS_FEATS = [
    ("rsi_14",             "RSI(14)"),
    ("return_1",           "1-bar Return"),
    ("vol_ratio_10",       "Volume Ratio(10)"),
    ("bb_pct_20",          "Bollinger %B(20)"),
    ("atr_pct_14",         "ATR %(14)"),
    ("macd_hist_12_26_9",  "MACD Histogram"),
]

# Filter to only features that exist
_VIS_FEATS = [(f, l) for f, l in _VIS_FEATS if f in FEATURE_COLS]

if _VIS_FEATS:
    pdf = df_clean_5b.select(
        [f for f, _ in _VIS_FEATS] + ["target"]
    ).to_pandas()
    pdf["target_label"] = pdf["target"].map({0: "DOWN (0)", 1: "UP (1)"})

    fig4 = make_subplots(
        rows=2, cols=3,
        subplot_titles=[lbl for _, lbl in _VIS_FEATS],
        vertical_spacing=0.12, horizontal_spacing=0.06,
    )

    for idx, (feat, label) in enumerate(_VIS_FEATS):
        row, col = divmod(idx, 3)
        row += 1; col += 1
        lo, hi = pdf[feat].quantile([0.01, 0.99])
        vis = pdf[(pdf[feat] >= lo) & (pdf[feat] <= hi)]

        for cls, color, side, name in [
            ("DOWN (0)", _DOWN_RED, "negative", "DOWN (0)"),
            ("UP (1)",   _UP_GREEN, "positive", "UP (1)"),
        ]:
            sub = vis.loc[vis["target_label"] == cls, feat]
            fig4.add_trace(
                go.Violin(
                    y=sub, name=name, side=side,
                    line_color=color, fillcolor=color,
                    opacity=0.55,
                    meanline_visible=True,
                    box_visible=True,
                    box_fillcolor="rgba(255,255,255,0.08)",
                    showlegend=(idx == 0),
                    scalemode="width",
                    width=0.9,
                ),
                row=row, col=col,
            )
        # Median annotations
        for cls, xoff in [("DOWN (0)", 0.7), ("UP (1)", 0.3)]:
            med = vis.loc[vis["target_label"] == cls, feat].median()
            fig4.add_annotation(
                text=f"{med:.5f}", x=xoff, y=med,
                xref=f"x{idx+1}" if idx > 0 else "x",
                yref=f"y{idx+1}" if idx > 0 else "y",
                showarrow=False,
                font=dict(size=8, color=_TEXT),
            )
        fig4.update_yaxes(title_text=feat, title_font=dict(size=9, color=_TEXT_MUT),
                          gridcolor=_GRID, tickfont=dict(color=_TEXT_MUT),
                          row=row, col=col)
        fig4.update_xaxes(showticklabels=False, row=row, col=col)

    # Style subplot titles to light color
    for ann in fig4.layout.annotations:
        ann.font.color = _TEXT

    fig4.update_layout(
        **_LAYOUT_KW,
        title=dict(text="Feature Distributions by Target Class",
                   font=dict(size=15, color=_TEXT)),
        width=1100, height=700,
        legend=dict(orientation="h", yanchor="bottom", y=1.02,
                    xanchor="center", x=0.5, font=dict(size=11, color=_TEXT)),
    )
    fig4.show()

# ═══════════════════════════════════════════════════════════════
# §5. TIME-SERIES SANITY PLOT
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§5. Time-Series Sanity Plot")
print(f"{'─' * 70}")

# Subsample for clean rendering (~2000 points)
step = max(1, len(df_clean_5b) // 2000)
df_vis = df_clean_5b.gather_every(step)

ts      = df_vis["minute_bucket"].to_list()
close_v = df_vis["close"].to_list()
tgt_arr = df_vis["target"].to_numpy()

# Check if SMA features exist (they don't - we have sma_dist_X)
# Use close with overlaid target instead
fig5 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.75, 0.25], vertical_spacing=0.03,
)

# ── Top: Close price line ────────────────────────────────────
fig5.add_trace(go.Scatter(
    x=ts, y=close_v, mode="lines",
    line=dict(color=_TEXT, width=1),
    name="Close", opacity=0.85,
), row=1, col=1)

# Target scatter overlay on price
up_idx = np.where(tgt_arr == 1)[0]
dn_idx = np.where(tgt_arr == 0)[0]
fig5.add_trace(go.Scatter(
    x=[ts[i] for i in up_idx],
    y=[close_v[i] for i in up_idx],
    mode="markers", marker=dict(color=_UP_GREEN, size=3, opacity=0.4),
    name="UP (1)",
), row=1, col=1)
fig5.add_trace(go.Scatter(
    x=[ts[i] for i in dn_idx],
    y=[close_v[i] for i in dn_idx],
    mode="markers", marker=dict(color=_DOWN_RED, size=3, opacity=0.4),
    name="DOWN (0)",
), row=1, col=1)

# ── Bottom: Target timeline ──────────────────────────────────
tgt_list = tgt_arr.tolist()
fig5.add_trace(go.Bar(
    x=ts, y=[1 if t == 1 else 0 for t in tgt_list],
    marker_color=[_UP_GREEN if t == 1 else _DOWN_RED for t in tgt_list],
    marker_line_width=0,
    opacity=0.6, showlegend=False,
    hovertemplate="%{x}<br>Target: %{y}<extra></extra>",
), row=2, col=1)

fig5.update_layout(
    **_LAYOUT_KW,
    title=dict(text="BTC/USDT — Close Price + Triple Barrier Labels",
               font=dict(size=14, color=_TEXT)),
    width=1200, height=650,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="center", x=0.5,
                font=dict(size=10, color=_TEXT)),
    hovermode="x unified",
)
fig5.update_yaxes(title_text="Price (USDT)", gridcolor=_GRID,
                  tickfont=dict(color=_TEXT_MUT), row=1, col=1)
fig5.update_yaxes(title_text="Target", gridcolor=_GRID,
                  tickvals=[0, 1], ticktext=["DOWN", "UP"],
                  tickfont=dict(color=_TEXT_MUT), row=2, col=1)
fig5.update_xaxes(gridcolor=_GRID, tickfont=dict(color=_TEXT_MUT),
                  row=1, col=1)
fig5.update_xaxes(title_text="Time", gridcolor=_GRID,
                  tickfont=dict(color=_TEXT_MUT), row=2, col=1)
fig5.show()

print(f"  Subsampled every {step}th bar → {len(df_vis):,} points")
print(f"  Time range: {ts[0]} → {ts[-1]}")

# ═══════════════════════════════════════════════════════════════
# §6. FEATURE DROP DECISIONS
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§6. Feature Drop Decisions (from §1 correlation analysis)")
print(f"{'─' * 70}")

_DROP: dict[str, str] = {}

for a, b, r in high_corr_pairs:
    if a in _DROP or b in _DROP:
        continue

    pair = {a, b}
    # Priority: keep features with higher |correlation with target|
    ar_target = abs(feat_target_corr.get(a, 0))
    br_target = abs(feat_target_corr.get(b, 0))

    if ar_target >= br_target:
        drop_col = b
        keep_col = a
    else:
        drop_col = a
        keep_col = b

    _DROP[drop_col] = f"corr({keep_col}) = {r:+.3f}"

features_to_drop = list(_DROP.keys())

print(f"  Features to drop: {len(features_to_drop)}")
if features_to_drop:
    print(f"  {'Feature':<30} {'Reason':}")
    print(f"  {'─'*30} {'─'*40}")
    for feat, reason in _DROP.items():
        print(f"  {feat:<30} {reason}")
else:
    print(f"  ✓ No highly correlated features to drop")

print(f"\n{'=' * 70}")
print("PHASE 5B SUMMARY")
print(f"{'=' * 70}")
print(f"  Total features:        {len(FEATURE_COLS)}")
print(f"  Features to drop:      {len(features_to_drop)}")
print(f"  Features retained:     {len(FEATURE_COLS) - len(features_to_drop)}")
print(f"  Primary metric:        {_PRIMARY_METRIC}")
print(f"  Class balance:         DOWN={100*n0/n_all:.1f}%, UP={100*n1/n_all:.1f}%")
print(f"\n  → features_to_drop exported for Phase 6A")
print(f"{'=' * 70}")

In [0]:
# PHASE 6A — Prepare Final Dataset for AutoML
# ============================================================
# Cleans the feature DataFrame, applies decisions from Phase 5
# EDA, and saves a Delta table ready for AutoML consumption.
#
# INPUT:  df_labeled   (Polars, ~40K × 105, Phase 4B)
#         FEATURE_COLS (list[str], 84 features, Phase 5A)
# OUTPUT: Delta table  → smart-trading-dev.02_silver.btcusdt_features_1min
#         (features + minute_bucket + target)
#
# WRITE STRATEGY:
#   mode('overwrite') + overwriteSchema=true allows full refresh
#   with schema evolution.  Safety is provided by:
#     1. Pre-write version checkpoint (logged rollback path)
#     2. Extended retention (30 days via table property)
#     3. Run metadata as table properties (audit trail)
#     4. Table comment documenting the asset
#   To recover: RESTORE TABLE ... TO VERSION AS OF <n>
# ============================================================

import polars as pl
import numpy as np

print("=" * 70)
print("PHASE 6A: PREPARE FINAL DATASET FOR AUTOML")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# §0. VERIFY & RE-ESTABLISH DEPENDENCIES
# ═══════════════════════════════════════════════════════════════
# Defensive: retrieve features_to_drop from Phase 5B if available,
# otherwise use a safe default. Filter to only valid features.

# Sanity gates — fail fast if upstream cells need re-running
assert "df_labeled" in dir(), "df_labeled not found — run Cells 9→18 first"
assert "FEATURE_COLS" in dir(), "FEATURE_COLS not found — run Cell 17 (Phase 5A) first"
assert len(FEATURE_COLS) == 84, f"Expected 84 features, got {len(FEATURE_COLS)}"

print(f"  df_labeled:       {len(df_labeled):,} rows × {len(df_labeled.columns)} cols")
print(f"  FEATURE_COLS:     {len(FEATURE_COLS)} features")

# Retrieve features_to_drop from Phase 5B (if available)
if "features_to_drop" in dir() and features_to_drop:
    # Filter to only features that exist in FEATURE_COLS
    features_to_drop_valid = [f for f in features_to_drop if f in FEATURE_COLS]
    
    if len(features_to_drop_valid) < len(features_to_drop):
        n_invalid = len(features_to_drop) - len(features_to_drop_valid)
        print(f"  ⚠ Filtered {n_invalid} features from drop list (not in FEATURE_COLS)")
    
    features_to_drop = features_to_drop_valid
else:
    # Fallback: minimal drop list if Phase 5B wasn't run
    features_to_drop = [f for f in ["atr_14", "return_1"] if f in FEATURE_COLS]
    print(f"  ⚠ Using fallback features_to_drop (run Cell 18/Phase 5B for full list)")

print(f"  features_to_drop: {len(features_to_drop)} correlated features")

# Build the surviving features (FEATURE_COLS minus drops)
_drop_set = set(features_to_drop)
final_features: list[str] = [c for c in FEATURE_COLS if c not in _drop_set]
expected_final = len(FEATURE_COLS) - len(features_to_drop)
assert len(final_features) == expected_final, \
    f"Expected {expected_final} surviving features, got {len(final_features)}"
print(f"  final_features:   {len(final_features)} features ({len(FEATURE_COLS)} - {len(features_to_drop)})")

# ═══════════════════════════════════════════════════════════════
# §1. DROP WARM-UP NULLS
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print("§1. Drop Warm-Up Nulls")
print(f"{'-' * 70}")

n_before = len(df_labeled)
df_clean = df_labeled.drop_nulls(subset=FEATURE_COLS)
n_after_nulls = len(df_clean)

print(f"  Before: {n_before:,}  →  After: {n_after_nulls:,}  (dropped {n_before - n_after_nulls})")

# Verify: zero nulls remaining in feature columns
null_sums = df_clean.select(FEATURE_COLS).null_count().row(0)
total_nulls = sum(null_sums)
assert total_nulls == 0, f"Still have {total_nulls} nulls in feature columns!"
print(f"  ✓ Null check passed (0 nulls across all {len(FEATURE_COLS)} feature columns)")

# ═══════════════════════════════════════════════════════════════
# §2. HANDLE INFINITE VALUES
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print("§2. Handle Infinite Values")
print(f"{'-' * 70}")

float_features = [
    c for c in FEATURE_COLS
    if df_clean[c].dtype in (pl.Float32, pl.Float64)
]

n_inf = sum(df_clean[c].is_infinite().sum() for c in float_features)
print(f"  Float feature columns: {len(float_features)}")
print(f"  Infinite values found: {n_inf}")

if n_inf > 0:
    df_clean = df_clean.with_columns(
        [
            pl.when(pl.col(c).is_infinite())
            .then(None)
            .otherwise(pl.col(c))
            .alias(c)
            for c in float_features
        ]
    ).drop_nulls(subset=float_features)
    print(f"  After inf removal: {len(df_clean):,} rows (dropped {n_after_nulls - len(df_clean)})")
else:
    print(f"  ✓ No infinities — skipping")

n_after_inf = len(df_clean)

# ═══════════════════════════════════════════════════════════════
# §3. SELECT FINAL COLUMNS (implicit drop of everything else)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print(f"§3. Select Final Columns ({len(final_features)} features + minute_bucket + target)")
print(f"{'-' * 70}")

final_columns = ["minute_bucket"] + final_features + ["target"]
df_clean = df_clean.select(final_columns)

print(f"  Columns selected: {len(final_columns)}")
print(f"  Features: {len(final_features)}")
print(f"  + minute_bucket (time column for AutoML split)")
print(f"  + target (label)")

# ═══════════════════════════════════════════════════════════════
# §4. CAST TARGET TO STRING
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print("§4. Cast Target to String (AutoML requirement)")
print(f"{'-' * 70}")

print(f"  Before cast — dtype: {df_clean['target'].dtype}")
df_clean = df_clean.with_columns(
    pl.col("target").cast(pl.Utf8).alias("target")
)

target_dist = df_clean["target"].value_counts().sort("target")
print(f"  After cast  — dtype: {df_clean['target'].dtype}")
print(f"  Distribution:")
for row in target_dist.iter_rows():
    label, count = row[0], row[1]
    pct = count / len(df_clean) * 100
    tag = "DOWN (SL/vertical)" if label == "0" else "UP (take-profit)"
    print(f"    '{label}': {count:>6,}  ({pct:.1f}%)  — {tag}")

# ═══════════════════════════════════════════════════════════════
# §5. TIME-BASED SPLIT DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print("§5. Time-Based Split Diagnostics (for reference)")
print(f"{'-' * 70}")

n_total = len(df_clean)
ts_sorted = df_clean["minute_bucket"].sort()

i_60 = int(n_total * 0.60)
i_80 = int(n_total * 0.80)

print(f"  Total rows:   {n_total:,}")
print(f"  Date range:   {ts_sorted[0]}  →  {ts_sorted[-1]}")
print(f"  ── AutoML 60/20/20 chronological split (approximate) ──")
print(f"  Train (~60%): {i_60:,} rows   up to ~{ts_sorted[i_60]}")
print(f"  Val   (~20%): {i_80 - i_60:,} rows   up to ~{ts_sorted[i_80]}")
print(f"  Test  (~20%): {n_total - i_80:,} rows   from  ~{ts_sorted[i_80]}")

# ═══════════════════════════════════════════════════════════════
# §6. CONVERT TO SPARK AND SAVE TO DELTA (hardened + schema evolution)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print("§6. Convert to Spark and Save to Delta (hardened write)")
print(f"{'-' * 70}")

from datetime import datetime, timezone
import hashlib

TABLE_NAME = f"`{CATALOG}`.`{SILVER}`.`btcusdt_features_1min`"
run_ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
feature_hash = hashlib.md5(
    ",".join(sorted(final_features)).encode()
).hexdigest()[:12]

# ── 6a. Pre-write: checkpoint current version ─────────────────
prev_version = None
try:
    hist = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME} LIMIT 1").collect()
    if hist:
        prev_version = hist[0]["version"]
        prev_ts = hist[0]["timestamp"]
        prev_count = spark.table(TABLE_NAME).count()
        print(f"  Pre-write checkpoint:")
        print(f"    Current version:  {prev_version}")
        print(f"    Current rows:     {prev_count:,}")
        print(f"    Last modified:    {prev_ts}")
        print(f"    Rollback command: RESTORE TABLE {TABLE_NAME}")
        print(f"                      TO VERSION AS OF {prev_version}")
except Exception:
    print(f"  Table does not exist yet — first write")

# ── 6b. Write with overwrite + schema evolution ───────────────
# CRITICAL FIX: Add overwriteSchema=true to allow schema changes
df_spark = spark.createDataFrame(df_clean.to_arrow())
df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"  ✓ Write completed with schema evolution enabled")

# ── 6c. Extend retention + tag with run metadata ─────────────
try:
    nb_path = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().notebookPath().get()
    )
except Exception:
    nb_path = "unknown"

spark.sql(f"""
    ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (
        'delta.deletedFileRetentionDuration' = 'interval 30 days',
        'pipeline.run_timestamp'             = '{run_ts}',
        'pipeline.row_count'                 = '{len(df_clean)}',
        'pipeline.feature_count'             = '{len(final_features)}',
        'pipeline.feature_hash'              = '{feature_hash}',
        'pipeline.source_notebook'           = '{nb_path}'
    )
""")

spark.sql(f"""
    COMMENT ON TABLE {TABLE_NAME} IS
    'BTCUSDT 1-min feature table for AutoML classification. """
    f"""{len(final_features)} technical indicator features + target (triple barrier label). """
    f"""Overwrite-safe: Delta time travel retained for 30 days.'
""")

# ── 6d. Post-write verification ───────────────────────────────
saved_count = spark.table(TABLE_NAME).count()
new_hist = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME} LIMIT 1").collect()
new_version = new_hist[0]["version"] if new_hist else "?"

print(f"\n  ✓ Saved {saved_count:,} rows to {TABLE_NAME}")
print(f"  ✓ Delta version:      {new_version}")
print(f"  ✓ Retention:          30 days (extended from default 7)")
print(f"  ✓ Feature hash:       {feature_hash}")
print(f"  ✓ Run metadata:       tagged in TBLPROPERTIES")
if prev_version is not None:
    print(f"  ✓ Rollback available: RESTORE TABLE {TABLE_NAME}")
    print(f"                        TO VERSION AS OF {prev_version}")

# ═══════════════════════════════════════════════════════════════
# §7. FINAL VERIFICATION & LEAKAGE ASSERTIONS
# ═══════════════════════════════════════════════════════════════
print(f"\n{'-' * 70}")
print("§7. Final Verification & Leakage Assertions")
print(f"{'-' * 70}")

df_verify = spark.table(TABLE_NAME)
schema_cols = [f.name for f in df_verify.schema.fields]

# LEAKAGE CHECK 1: no tb_* columns (triple barrier metadata)
tb_leaked = [c for c in schema_cols if c.startswith("tb_")]
assert len(tb_leaked) == 0, f"LEAKAGE! tb_* columns found: {tb_leaked}"
print(f"  ✓ Leakage check #1 passed — no tb_* columns")

# LEAKAGE CHECK 2: no raw price columns
raw_prices = {"open", "high", "low", "close", "vwap"}
raw_leaked = [c for c in schema_cols if c in raw_prices]
assert len(raw_leaked) == 0, f"Raw price columns found: {raw_leaked}"
print(f"  ✓ Leakage check #2 passed — no raw price columns")

# SCHEMA CHECK: expected column count
n_schema = len(schema_cols)
n_features_saved = n_schema - 2  # minus minute_bucket and target
assert n_features_saved == len(final_features), \
    f"Expected {len(final_features)} features in table, got {n_features_saved}"
print(f"  ✓ Schema check passed — {n_schema} columns ({n_features_saved} features + minute_bucket + target)")

# Print final schema
print(f"\n  Final Schema:")
df_verify.printSchema()

# Display sample
print(f"  Sample rows:")
display(df_verify.orderBy("minute_bucket").limit(5))

# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PHASE 6A SUMMARY")
print("=" * 70)
print(f"  Input:           {n_before:,} rows × {len(df_labeled.columns)} cols")
print(f"  Warm-up dropped: {n_before - n_after_nulls} rows")
print(f"  Infinities:      {n_inf} values replaced")
print(f"  Correlated:      {len(features_to_drop)} features dropped")
print(f"  Non-features:    raw OHLCV + TB metadata + flags excluded via SELECT")
print(f"  Target cast:     Int → String ('0'/'1')")
print(f"  Output:          {saved_count:,} rows × {n_schema} cols")
print(f"  Delta table:     {TABLE_NAME}")
print(f"  Delta version:   {new_version}")
print(f"  Retention:       30 days (time travel safe)")
print(f"  Feature hash:    {feature_hash}")
print(f"  Primary metric:  roc_auc (balanced classes: ~50/50)")
if prev_version is not None:
    print(f"  Rollback:        RESTORE TABLE {TABLE_NAME}")
    print(f"                   TO VERSION AS OF {prev_version}")
print(f"\n{'=' * 70}")
print(f"✓ Phase 6A complete — proceed to Phase 6B (Baselines)")
print(f"{'=' * 70}")

In [0]:
# PHASE 6B — Establish Baseline Models via btcusdt_utils
# ============================================================
# Uses btcusdt_utils.evaluation.compute_baselines() for 5 naive
# baselines. Replaces ~200 lines of inline computation.
#
# INPUT:  Delta table from Phase 6A
# OUTPUT: baseline_results (dict) → Phase 8A comparison
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Import viz theme only (not CATALOG/SILVER)
from btcusdt_utils import (
    _BG, _BG_CARD, _TEXT, _TEXT_MUT, _GRID,
    _BLUE, _GREEN, _ORANGE, _PURPLE, _CYAN, _RED,
    _FONT,
)
from btcusdt_utils.evaluation import compute_baselines, compute_metrics

print("=" * 70)
print("PHASE 6B: ESTABLISH BASELINE MODELS (via btcusdt_utils)")
print("=" * 70)

# ── Load & Split (80/20 chronological) ─────────────────────────────
TABLE_REF = f"`{CATALOG}`.`{SILVER}`.`btcusdt_features_1min`"
df_baseline = spark.table(TABLE_REF).orderBy("minute_bucket").toPandas()

n_total = len(df_baseline)
i_split = int(n_total * 0.80)
y_train = df_baseline.iloc[:i_split]["target"].astype(int).values
y_test  = df_baseline.iloc[i_split:]["target"].astype(int).values

print(f"  Total rows: {n_total:,}")
print(f"  Train (80%): {len(y_train):,}")
print(f"  Test  (20%): {len(y_test):,}")

# ── Compute all 5 baselines in one call ─────────────────────────
baseline_results = compute_baselines(y_test, y_train)

print(f"\n  Baselines computed:")
for bl in baseline_results.values():
    print(f"    {bl.name:<22} acc={bl.metrics['accuracy']:.3f}  "
          f"f1={bl.metrics['f1']:.3f}  auc={bl.metrics['roc_auc']:.3f}")

# ── Visualization ───────────────────────────────────────────────
colors = [_BLUE, _GREEN, _ORANGE, _PURPLE, _CYAN]
bl_names = list(baseline_results.keys())
bl_accs  = [baseline_results[n].metrics["accuracy"] for n in bl_names]
bl_f1s   = [baseline_results[n].metrics["f1"] for n in bl_names]
bl_aucs  = [baseline_results[n].metrics["roc_auc"] for n in bl_names]

_LAYOUT_KW = dict(template="plotly_dark", plot_bgcolor=_BG,
                  paper_bgcolor=_BG_CARD, font=_FONT)

fig = go.Figure()
for i, (name, acc, f1, auc, color) in enumerate(
    zip(bl_names, bl_accs, bl_f1s, bl_aucs, colors)
):
    fig.add_trace(go.Bar(
        name=name, x=["Accuracy", "F1", "ROC AUC"],
        y=[acc, f1, auc], marker_color=color,
        text=[f"{acc:.3f}", f"{f1:.3f}", f"{auc:.3f}"],
        textposition="outside",
        textfont=dict(size=10, color=_TEXT),
    ))

fig.update_layout(
    **_LAYOUT_KW,
    title=dict(text="Baseline Model Comparison", font=dict(size=14, color=_TEXT)),
    xaxis=dict(gridcolor=_GRID, tickfont=dict(color=_TEXT_MUT)),
    yaxis=dict(title="Score", range=[0, 1.05], gridcolor=_GRID,
               tickfont=dict(color=_TEXT_MUT)),
    barmode="group",
    width=900, height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="center", x=0.5, font=dict(size=10, color=_TEXT)),
)
fig.show()

print("\n" + "=" * 70)
print("✓ Phase 6B complete — proceed to Phase 7A (Dependencies)")
print("=" * 70)

In [0]:
# PHASE 7A — ML Dependencies (already installed in Cell 8)
# ============================================================
# All ML packages (lightgbm, xgboost, optuna, scikit-learn)
# are now installed in Cell 8 (Phase 0) alongside polars.
# This cell is kept as a no-op for notebook structure.
# ============================================================
print("\u2713 ML dependencies already installed in Cell 8 (Phase 0)")
print("  No additional installation or kernel restart needed.")

In [0]:
# PHASE 7B-A — Load Training Data from Delta Table
# ============================================================
# INPUT:  Delta table from Phase 6A
# OUTPUT: df_full (Pandas DataFrame with all features)
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE 7B-A: LOAD TRAINING DATA")
print("=" * 70)

TABLE_REF = f"`{CATALOG}`.`{SILVER}`.`btcusdt_features_1min`"
print(f"\nLoading: {TABLE_REF}")

df_full = spark.table(TABLE_REF).orderBy("minute_bucket").toPandas()

print(f"  Total rows:    {len(df_full):,}")
print(f"  Total columns: {len(df_full.columns)}")
print(f"  Date range:    {df_full['minute_bucket'].min()} → {df_full['minute_bucket'].max()}")

print("\n" + "=" * 70)
print("✓ Phase 7B-A complete — proceed to Phase 7B-B (Split)")
print("=" * 70)

In [0]:
# PHASE 7B-B — Chronological Train/Val/Test Split
# ============================================================
# INPUT:  df_full (from Phase 7B-A)
# OUTPUT: df_train, df_val, df_test, feature_cols
# ============================================================

import numpy as np

print("=" * 70)
print("PHASE 7B-B: CHRONOLOGICAL SPLIT")
print("=" * 70)

n_total = len(df_full)
i_train_end = int(n_total * TRAIN_RATIO)
i_val_end = int(n_total * (TRAIN_RATIO + VAL_RATIO))

df_train = df_full.iloc[:i_train_end].copy()
df_val   = df_full.iloc[i_train_end:i_val_end].copy()
df_test  = df_full.iloc[i_val_end:].copy()

feature_cols = [c for c in df_full.columns if c not in ("minute_bucket", "target")]

print(f"\nSplit configuration: {TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%}")
print(f"  Total:    {n_total:,} rows")
print(f"  Train:    {len(df_train):,} rows ({TRAIN_RATIO:.0%})")
print(f"  Val:      {len(df_val):,} rows ({VAL_RATIO:.0%})")
print(f"  Test:     {len(df_test):,} rows ({TEST_RATIO:.0%})")
print(f"  Features: {len(feature_cols)}")

print("\n" + "=" * 70)
print("✓ Phase 7B-B complete — proceed to Phase 7B-C (Scaling)")
print("=" * 70)

In [0]:
# PHASE 7B-C — Feature Scaling (StandardScaler)
# ============================================================
# INPUT:  df_train, df_val, df_test, feature_cols (from 7B-B)
# OUTPUT: X_train, X_val, X_test, y_train, y_val, y_test
#         X_train_scaled, X_val_scaled, X_test_scaled, scaler
# ============================================================

from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("PHASE 7B-C: FEATURE SCALING")
print("=" * 70)

# Extract features and targets
X_train = df_train[feature_cols].values
X_val   = df_val[feature_cols].values
X_test  = df_test[feature_cols].values
y_train = df_train["target"].astype(int).values
y_val   = df_val["target"].astype(int).values
y_test  = df_test["target"].astype(int).values

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"\nScaled feature statistics (train set):")
print(f"  Mean:  {X_train_scaled.mean():.6f}  (should be ~0)")
print(f"  Std:   {X_train_scaled.std():.6f}   (should be ~1)")
print(f"  Min:   {X_train_scaled.min():.2f}")
print(f"  Max:   {X_train_scaled.max():.2f}")

print("\n" + "=" * 70)
print("✓ Phase 7B-C complete — proceed to Phase 7B-D (Training)")
print("=" * 70)

In [0]:
# PHASE 7B-D — Train 5 Models with Optuna Hyperparameter Tuning
# ============================================================
# INPUT:  X_train, X_val, X_test (raw + scaled), y_train, y_val, y_test,
#         feature_cols (from Phase 7B-C)
# OUTPUT: model_results, best_model, best_model_name, best_run_id
# ============================================================

import mlflow
from btcusdt_utils.training import train_and_evaluate

print("=" * 70)
print("PHASE 7B-D: TRAIN MODELS WITH OPTUNA")
print("=" * 70)

print(f"\nTraining configuration:")
print(f"  Experiment:     {EXPERIMENT_NAME}")
print(f"  Optuna trials:  {OPTUNA_TRIALS}")
print(f"  Timeout:        {OPTUNA_TIMEOUT_SEC}s")
print(f"  Algorithms:     LightGBM, XGBoost, RandomForest, LogisticRegression, DecisionTree")

print(f"\nStarting hyperparameter tuning...\n")

model_results, best_model, best_model_name, best_run_id = train_and_evaluate(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    X_train_scaled, X_val_scaled, X_test_scaled,
    feature_cols,
    experiment_name=EXPERIMENT_NAME,
    n_trials=OPTUNA_TRIALS,
    timeout=OPTUNA_TIMEOUT_SEC,
)

print(f"\n{'=' * 70}")
print(f"TRAINING COMPLETE")
print(f"{'=' * 70}")
print(f"\n  Best model:  {best_model_name}")
print(f"  Best run ID: {best_run_id}")

print("\nTest set metrics:")
for name, result in model_results.items():
    m = result.test_metrics
    marker = " ★" if name == best_model_name else "  "
    print(f"{marker} {name:<22} acc={m['accuracy']:.3f}  f1={m['f1']:.3f}  auc={m['roc_auc']:.3f}")

print("\n" + "=" * 70)
print("✓ Phase 7B-D complete — proceed to Phase 7B-E (Visualization)")
print("=" * 70)

In [0]:
# PHASE 7B-E — Model Comparison Visualization
# ============================================================
# INPUT:  model_results, best_model_name (from Phase 7B-D)
# OUTPUT: Grouped bar chart comparing all models
# ============================================================

import plotly.graph_objects as go
from btcusdt_utils import (
    _BG, _BG_CARD, _TEXT, _TEXT_MUT, _GRID,
    _BLUE, _GREEN, _ORANGE, _PURPLE,
    _FONT,
)

print("=" * 70)
print("PHASE 7B-E: MODEL COMPARISON VISUALIZATION")
print("=" * 70)

# Extract metrics for all models
model_names = list(model_results.keys())
accs = [model_results[n].test_metrics["accuracy"] for n in model_names]
f1s  = [model_results[n].test_metrics["f1"] for n in model_names]
aucs = [model_results[n].test_metrics["roc_auc"] for n in model_names]

_LAYOUT_KW = dict(template="plotly_dark", plot_bgcolor=_BG,
                  paper_bgcolor=_BG_CARD, font=_FONT)

fig = go.Figure()
colors = [_BLUE, _GREEN, _ORANGE, _PURPLE, (_PURPLE if len(model_names) > 4 else _ORANGE)]

for i, (name, acc, f1, auc) in enumerate(zip(model_names, accs, f1s, aucs)):
    color = colors[i] if i < len(colors) else _BLUE
    marker = " ★" if name == best_model_name else ""
    fig.add_trace(go.Bar(
        name=f"{name}{marker}",
        x=["Accuracy", "F1", "ROC AUC"],
        y=[acc, f1, auc],
        marker_color=color,
        text=[f"{acc:.3f}", f"{f1:.3f}", f"{auc:.3f}"],
        textposition="outside",
        textfont=dict(size=10, color=_TEXT),
    ))

fig.update_layout(
    **_LAYOUT_KW,
    title=dict(text="Model Comparison (Test Set)",
               font=dict(size=14, color=_TEXT)),
    xaxis=dict(gridcolor=_GRID, tickfont=dict(color=_TEXT_MUT)),
    yaxis=dict(title="Score", range=[0, 1.05], gridcolor=_GRID,
               tickfont=dict(color=_TEXT_MUT)),
    barmode="group",
    width=1000, height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="center", x=0.5,
                font=dict(size=10, color=_TEXT)),
)
fig.show()

print("\n" + "=" * 70)
print("✓ Phase 7B-E complete — proceed to Phase 8A (Evaluation)")
print("=" * 70)

In [0]:
# PHASE 8A-A — Generate Predictions from Best Model
# ============================================================
# INPUT:  best_model, best_model_name, X_test, X_test_scaled,
#         y_test (from Phase 7B-D)
# OUTPUT: y_pred, y_prob
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE 8A-A: GENERATE PREDICTIONS")
print("=" * 70)

print(f"\nBest model: {best_model_name}")

# LogisticRegression needs scaled features
use_scaled = best_model_name == "LogisticRegression"
X_te = X_test_scaled if use_scaled else X_test

print(f"  Using {'scaled' if use_scaled else 'raw'} features")

# Generate predictions
y_pred = best_model.predict(X_te)
y_prob = best_model.predict_proba(X_te)[:, 1]

print(f"\nPrediction statistics:")
print(f"  Test samples:     {len(y_test):,}")
print(f"  Predictions:      {len(y_pred):,}")
print(f"  P(UP) range:      [{y_prob.min():.4f}, {y_prob.max():.4f}]")
print(f"  P(UP) mean:       {y_prob.mean():.4f}")
print(f"  Predicted UP:     {(y_pred == 1).sum():,} ({(y_pred == 1).mean():.1%})")
print(f"  Predicted DOWN:   {(y_pred == 0).sum():,} ({(y_pred == 0).mean():.1%})")

print("\n" + "=" * 70)
print("✓ Phase 8A-A complete — proceed to Phase 8A-B (Baseline Comparison)")
print("=" * 70)

In [0]:
# PHASE 8A-B — Baseline Comparison Table (via btcusdt_utils)
# ============================================================
# INPUT:  y_test, y_train, model_results (from 8A-A)
# OUTPUT: Comparison table of baselines vs ML models
# ============================================================

import numpy as np
import pandas as pd
from btcusdt_utils.evaluation import compute_baselines, build_comparison_table

print("=" * 70)
print("PHASE 8A-B: BASELINE COMPARISON")
print("=" * 70)

# Compute all baselines using btcusdt_utils (replaces ~50 lines of manual code)
print("\nComputing baselines via btcusdt_utils.evaluation.compute_baselines()...")
baseline_results = compute_baselines(y_test, y_train)

print(f"\nBaselines computed:")
for bl in baseline_results.values():
    print(f"  {bl.name:<22} acc={bl.metrics['accuracy']:.3f}  "
          f"f1={bl.metrics['f1']:.3f}  auc={bl.metrics['roc_auc']:.3f}")

# Build comparison table (baselines + ML models)
print(f"\nML Models (from Phase 7B-D):")
best_tm = model_results[best_model_name].test_metrics

for name, res in model_results.items():
    tm = res.test_metrics
    marker = " ★" if name == best_model_name else "  "
    print(f"{marker} {name:<22} acc={tm['accuracy']:.3f}  "
          f"f1={tm['f1']:.3f}  auc={tm['roc_auc']:.3f}")

# Combine baselines + models for display
comparison_rows = []

# Add baselines
for bl_key, bl in baseline_results.items():
    comparison_rows.append({
        "Model": bl.name,
        "Type": "Baseline",
        "Accuracy": f"{bl.metrics['accuracy']:.1%}",
        "F1": f"{bl.metrics['f1']:.3f}",
        "ROC-AUC": f"{bl.metrics['roc_auc']:.3f}",
    })

# Add ML models
for name, res in model_results.items():
    tm = res.test_metrics
    tag = " ★" if name == best_model_name else ""
    comparison_rows.append({
        "Model": f"{name}{tag}",
        "Type": "ML Model",
        "Accuracy": f"{tm['accuracy']:.1%}",
        "F1": f"{tm['f1']:.3f}",
        "ROC-AUC": f"{tm['roc_auc']:.3f}",
    })

df_comparison = pd.DataFrame(comparison_rows)
print("\nComparison Table:")
print(df_comparison.to_string(index=False))
display(spark.createDataFrame(df_comparison))

# Calculate lift vs best baseline
best_baseline_auc = max(bl.metrics['roc_auc'] for bl in baseline_results.values())
best_baseline_name = [bl.name for bl in baseline_results.values() 
                       if bl.metrics['roc_auc'] == best_baseline_auc][0]

acc_lift = (best_tm['accuracy'] - baseline_results['last_label'].metrics['accuracy']) / \
           baseline_results['last_label'].metrics['accuracy'] * 100
auc_lift = (best_tm['roc_auc'] - best_baseline_auc) / best_baseline_auc * 100

print(f"\nLift vs best baseline ('{best_baseline_name}'):")
print(f"  ROC-AUC lift: {auc_lift:+.1f}%")
print(f"  Accuracy:     {best_tm['accuracy']:.1%} vs {baseline_results['last_label'].metrics['accuracy']:.1%}")

print("\n" + "=" * 70)
print("✓ Phase 8A-B complete — proceed to Phase 8A-C (Confusion Matrix)")
print("=" * 70)

In [0]:
# PHASE 8A-C — Confusion Matrix + ROC/PR Curves (via btcusdt_utils)
# ============================================================
# INPUT:  y_test, y_pred, y_prob, best_model_name (from 8A-A)
# OUTPUT: Confusion matrix heatmap + ROC + PR curves
# ============================================================

import numpy as np
from sklearn.metrics import confusion_matrix
from btcusdt_utils.evaluation import (
    plot_confusion_matrix,
    plot_roc_curves,
    plot_precision_recall,
)

print("=" * 70)
print("PHASE 8A-C: CONFUSION MATRIX + ROC/PR CURVES")
print("=" * 70)

# ── 1. Confusion Matrix (using btcusdt_utils) ────────────────────
print("\n1. Confusion Matrix")

# Compute raw counts for display
cm = confusion_matrix(y_test, y_pred)
print(f"  TN={cm[0][0]:,}  FP={cm[0][1]:,}")
print(f"  FN={cm[1][0]:,}  TP={cm[1][1]:,}")
fpr_val = cm[0][1] / (cm[0][0] + cm[0][1])
print(f"  False Positive Rate: {fpr_val:.1%}")

# Plot using btcusdt_utils (replaces ~30 lines of manual Plotly code)
fig_cm = plot_confusion_matrix(
    y_test, y_pred,
    title=f"Confusion Matrix — {best_model_name}",
    labels=("DOWN (0)", "UP (1)"),
)
fig_cm.show()

# ── 2. ROC Curve (using btcusdt_utils) ─────────────────────────
print("\n2. ROC Curve")

# Plot using btcusdt_utils (replaces ~25 lines of manual Plotly code)
fig_roc = plot_roc_curves(
    y_test,
    model_probs={best_model_name: y_prob},
)
fig_roc.update_layout(title=f"ROC Curve — {best_model_name}")
fig_roc.show()

roc_auc_val = best_tm["roc_auc"]
print(f"  ROC-AUC: {roc_auc_val:.4f}")

# ── 3. Precision-Recall Curve (using btcusdt_utils) ───────────────
print("\n3. Precision-Recall Curve")

# Plot using btcusdt_utils (replaces ~25 lines of manual Plotly code)
fig_pr = plot_precision_recall(
    y_test,
    model_probs={best_model_name: y_prob},
)
fig_pr.update_layout(title=f"Precision-Recall Curve — {best_model_name}")
fig_pr.show()

from sklearn.metrics import average_precision_score
pr_auc = average_precision_score(y_test, y_prob)
print(f"  PR-AUC (Average Precision): {pr_auc:.4f}")

print("\n" + "=" * 70)
print("✓ Phase 8A-C complete — proceed to Phase 8A-D (Classification Report)")
print("=" * 70)

In [0]:
# PHASE 8A-D — Classification Report + Feature Importances (via btcusdt_utils)
# ============================================================
# INPUT:  y_test, y_pred, best_model, feature_cols (from 8A-A)
# OUTPUT: Classification report + top N feature importances
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from btcusdt_utils.evaluation import plot_feature_importance

print("=" * 70)
print("PHASE 8A-D: CLASSIFICATION REPORT + FEATURE IMPORTANCES")
print("=" * 70)

# ── 1. Classification Report ───────────────────────────────────
print("\n1. Classification Report\n")
print(classification_report(y_test, y_pred,
                            target_names=["DOWN", "UP"], digits=4))

# ── 2. Feature Importances (using btcusdt_utils) ────────────────────
print("\n2. Feature Importances")

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    imp_label = "Feature Importance"
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
    imp_label = "|Coefficient|"
else:
    importances = None
    imp_label = None

if importances is not None:
    # Plot using btcusdt_utils (replaces ~30 lines of manual Plotly code)
    fig_fi = plot_feature_importance(
        importances,
        feature_cols,
        top_n=TOP_N_FEATURES,
        title=f"Top {TOP_N_FEATURES} Features — {best_model_name} ({imp_label})",
    )
    fig_fi.show()

    # Print top 5 for reference
    fi_df = pd.DataFrame({
        "feature": feature_cols, "importance": importances,
    }).sort_values("importance", ascending=False)

    print(f"\nTop 5 features:")
    for _, row in fi_df.head(5).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.6f}")

    # Leakage check
    total_imp = importances.sum()
    top1_imp = fi_df["importance"].iloc[0]
    top1_name = fi_df["feature"].iloc[0]
    if total_imp > 0 and top1_imp > 0.5 * total_imp:
        print(f"\n  ⚠️ LEAKAGE WARNING: '{top1_name}' dominates "
              f"({top1_imp / total_imp:.1%} of total)")
    else:
        print(f"\n  ✓ No single feature dominates — no obvious leakage")
else:
    print("  (Feature importances not available for this model type)")

print("\n" + "=" * 70)
print("✓ Phase 8A-D complete — proceed to Phase 8A-E (Temporal Analysis)")
print("=" * 70)

In [0]:
# PHASE 8A-E — Probability Distribution + Rolling Accuracy Over Time
# ============================================================
# INPUT:  y_test, y_pred, y_prob, df_test (from 8A-A)
# OUTPUT: Probability histograms + rolling accuracy time series
# ============================================================

import numpy as np
import plotly.graph_objects as go
from btcusdt_utils import (
    _BG, _BG_CARD, _TEXT, _TEXT_MUT, _GRID,
    _RED, _GREEN, _BLUE, _ORANGE,
    _FONT,
)

print("=" * 70)
print("PHASE 8A-E: TEMPORAL ANALYSIS")
print("=" * 70)

_LAYOUT_KW = dict(template="plotly_dark", plot_bgcolor=_BG,
                  paper_bgcolor=_BG_CARD, font=_FONT)

# ── Probability Distribution ──────────────────────────────────
print("\n1. Prediction Probability Distribution")

fig_prob = go.Figure()
fig_prob.add_trace(go.Histogram(
    x=y_prob[y_test == 0], name="Actual DOWN",
    marker_color=_RED, opacity=0.7, nbinsx=50,
))
fig_prob.add_trace(go.Histogram(
    x=y_prob[y_test == 1], name="Actual UP",
    marker_color=_GREEN, opacity=0.7, nbinsx=50,
))
fig_prob.update_layout(
    **_LAYOUT_KW,
    title=dict(text=f"P(UP) Distribution by Actual Class — {best_model_name}",
              font=dict(size=16, color=_TEXT)),
    xaxis=dict(title="Predicted P(UP)", gridcolor=_GRID),
    yaxis=dict(title="Count", gridcolor=_GRID),
    barmode="overlay",
    legend=dict(x=0.02, y=0.98, font=dict(color=_TEXT)),
    width=900, height=450,
)
fig_prob.show()

mean_p_down = y_prob[y_test == 0].mean()
mean_p_up   = y_prob[y_test == 1].mean()
separation  = mean_p_up - mean_p_down
print(f"  Mean P(UP) | actual DOWN: {mean_p_down:.4f}")
print(f"  Mean P(UP) | actual UP:   {mean_p_up:.4f}")
print(f"  Separation:                {separation:.4f}")
if separation < 0.02:
    print("  ⚠️ Minimal separation — model struggles to distinguish classes")

# ── Rolling Accuracy ────────────────────────────────────────
print("\n2. Rolling Accuracy Over Time")

df_time = df_test[["minute_bucket"]].copy()
df_time["correct"] = (y_pred == y_test).astype(int)
df_time["rolling_acc"] = (
    df_time["correct"].rolling(window=ROLLING_WINDOW_SIZE, min_periods=50).mean()
)

fig_roll = go.Figure()
fig_roll.add_trace(go.Scatter(
    x=df_time["minute_bucket"], y=df_time["rolling_acc"],
    mode="lines", name=f"Rolling Accuracy (w={ROLLING_WINDOW_SIZE})",
    line=dict(color=_BLUE, width=1.5),
))
fig_roll.add_hline(
    y=0.5, line=dict(color=_TEXT_MUT, dash="dash", width=1),
    annotation_text="50% (random)",
    annotation=dict(font_size=10, font_color=_TEXT_MUT),
)
fig_roll.add_hline(
    y=BASELINE_LASTTB_ACC,
    line=dict(color=_ORANGE, dash="dash", width=1),
    annotation_text=f"Last TB Label ({BASELINE_LASTTB_ACC:.1%})",
    annotation=dict(font_size=10, font_color=_ORANGE, xanchor="left"),
)
fig_roll.update_layout(
    **_LAYOUT_KW,
    title=dict(text=f"Rolling Accuracy (window={ROLLING_WINDOW_SIZE}) — {best_model_name}",
              font=dict(size=16, color=_TEXT)),
    xaxis=dict(title="Time", gridcolor=_GRID),
    yaxis=dict(title="Accuracy", range=[0.3, 0.85], gridcolor=_GRID),
    width=1000, height=400,
)
fig_roll.show()

print("\n" + "=" * 70)
print("✓ Phase 8A-E complete — proceed to Phase 8A-F (Model Quality)")
print("=" * 70)

In [0]:
# PHASE 8A-F — Calibration Curve + MLflow Trial Comparison (via btcusdt_utils)
# ============================================================
# INPUT:  y_test, y_prob, best_model_name (from 8A-A)
# OUTPUT: Calibration plot + top MLflow runs table
# ============================================================

import numpy as np
import pandas as pd
import mlflow
from btcusdt_utils.evaluation import plot_calibration

print("=" * 70)
print("PHASE 8A-F: MODEL QUALITY ASSESSMENT")
print("=" * 70)

# ── 1. Calibration Curve (using btcusdt_utils) ──────────────────────
print("\n1. Calibration Curve")

# Plot using btcusdt_utils (replaces ~20 lines of manual Plotly code)
fig_cal = plot_calibration(
    y_test,
    model_probs={best_model_name: y_prob},
    n_bins=10,
)
fig_cal.update_layout(title=f"Calibration Curve — {best_model_name}")
fig_cal.show()

print("  Calibration measures how well predicted probabilities match observed frequencies.")
print("  Points near the diagonal indicate well-calibrated predictions.")

# ── 2. MLflow Trial Comparison ───────────────────────────────
print("\n2. MLflow Trial Comparison (Top Runs)")

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_roc_auc DESC"],
    max_results=10,
)

cols_to_show = [
    "run_id", "params.model_type",
    "metrics.val_roc_auc", "metrics.test_roc_auc",
    "metrics.test_accuracy", "metrics.test_f1",
]
available = [c for c in cols_to_show if c in runs_df.columns]
top_runs = runs_df[available].head(5).copy()
top_runs.columns = [c.split(".")[-1] for c in available]
print()
print(top_runs.to_string(index=False))
display(spark.createDataFrame(top_runs.astype(str)))

print("\n" + "=" * 70)
print("✓ Phase 8A-F complete — proceed to Phase 8A-G (Final Verdict)")
print("=" * 70)

In [0]:
# PHASE 8A-G — Final Verdict + Recommendations
# ============================================================
# INPUT:  best_model_name, best_tm (from 8A-A)
# OUTPUT: Final assessment + actionable recommendations
# ============================================================

print("=" * 70)
print("PHASE 8A-G: FINAL VERDICT")
print("=" * 70)

auc_val = best_tm["roc_auc"]
acc_val = best_tm["accuracy"]

# Verdict logic
if auc_val > 0.55:
    verdict = "MODEL SHOWS PREDICTIVE SIGNAL ABOVE RANDOM"
    icon = "✅"
    detail = "ROC-AUC > 0.55 indicates meaningful discrimination ability."
elif auc_val > 0.52:
    verdict = "MARGINAL SIGNAL — MAY NOT SURVIVE TRANSACTION COSTS"
    icon = "⚠️"
    detail = "ROC-AUC 0.52–0.55 is marginal. Signal likely eroded by spreads/fees."
else:
    verdict = "NO MEANINGFUL SIGNAL DETECTED"
    icon = "❌"
    detail = ("ROC-AUC ≤ 0.52 is near random chance. "
              "Features lack predictive power for 1-min direction.")

print(f"\n  {icon} {verdict}")
print(f"  {detail}")
print(f"\n  Best model: {best_model_name}")
print(f"  ROC-AUC:    {auc_val:.4f}")
print(f"  Accuracy:   {acc_val:.1%}")

beats_baseline = auc_val > BASELINE_LASTTB_AUC
if beats_baseline:
    print(f"\n  ✅ Beats 'Last TB Label' baseline ({BASELINE_LASTTB_AUC:.3f})")
else:
    print(f"\n  ❌ Does NOT beat 'Last TB Label' baseline ({BASELINE_LASTTB_AUC:.3f})")
    print(f"     Gap: {BASELINE_LASTTB_AUC - auc_val:.4f} AUC points")

print(f"\n  RECOMMENDATIONS:")
print(f"    1. Add diverse features (order book depth, funding rate, cross-asset)")
print(f"    2. Try longer prediction horizons (5-min, 15-min, 1-hour)")
print(f"    3. Consider regime-aware models (volatility clustering)")
print(f"    4. Increase training data (> 1 month)")
print(f"    5. Explore online learning for non-stationary dynamics")

print(f"\n{'=' * 70}")
print("✓ Phase 8A-G complete — ALL EVALUATION PHASES FINISHED")
print(f"{'=' * 70}")

In [0]:
# PHASE 8B — Register Best Model to Unity Catalog
# ============================================================
# INPUT:  best_model, best_model_name, model_results,
#         feature_cols, X_test*, y_test  (from Phase 7B)
# OUTPUT: Registered model in Unity Catalog with alias
#
# NOTE: Original model was logged without signature.
#       UC requires signature + input_example, so we re-log
#       the best model in a new run before registering.
# ============================================================

import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature

# Re-establish config (cell independence)
CATALOG = "smart_trading_dev"
GOLD    = "03_gold"
EXPERIMENT_NAME = "/Users/nthuan2109@gmail.com/btcusdt_direction_manual_ml"

print("=" * 70)
print("PHASE 8B: REGISTER BEST MODEL TO UNITY CATALOG")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# §1. REGISTRATION DECISION GATE
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§1. Registration Decision")
print(f"{'─' * 70}")

auc_val = model_results[best_model_name]["test_metrics"]["roc_auc"]
acc_val = model_results[best_model_name]["test_metrics"]["accuracy"]

if auc_val <= 0.52:
    print(f"  ⚠️  Model AUC ({auc_val:.4f}) is near random chance")
    print(f"     Registering as 'candidate' for tracking & comparison")
    alias = "candidate"
elif auc_val <= 0.708:
    print(f"  ⚠️  Model AUC ({auc_val:.4f}) doesn't beat baseline (0.708)")
    print(f"     Registering as 'candidate'")
    alias = "candidate"
else:
    print(f"  ✅ Model AUC ({auc_val:.4f}) beats baseline — registering as 'champion'")
    alias = "champion"

# ═══════════════════════════════════════════════════════════════
# §2. RE-LOG MODEL WITH SIGNATURE (required for UC)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§2. Re-log Model with Signature")
print(f"{'─' * 70}")

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

# Select correct test data for signature inference
use_scaled = best_model_name == "LogisticRegression"
X_te = X_test_scaled if use_scaled else X_test

# Build input DataFrame with feature names
X_input_df = pd.DataFrame(X_te[:5], columns=feature_cols)
y_pred_sample = best_model.predict(X_te[:5])

signature = infer_signature(
    model_input=X_input_df,
    model_output=y_pred_sample,
)
print(f"  Signature inferred:")
print(f"    Inputs:  {len(signature.inputs.inputs)} features")
print(f"    Outputs: {signature.outputs}")

# Log to new run with signature + input_example
import lightgbm as lgb
import xgboost as xgb

with mlflow.start_run(run_name=f"{best_model_name}_uc_ready") as run:
    mlflow.log_params(model_results[best_model_name]["best_params"])
    mlflow.log_param("model_type", best_model_name)
    for metric_name, metric_val in model_results[best_model_name]["test_metrics"].items():
        mlflow.log_metric(f"test_{metric_name}", metric_val)

    if best_model_name == "LightGBM":
        mlflow.lightgbm.log_model(
            best_model, artifact_path="model",
            signature=signature, input_example=X_input_df,
        )
    elif best_model_name == "XGBoost":
        mlflow.xgboost.log_model(
            best_model, artifact_path="model",
            signature=signature, input_example=X_input_df,
        )
    else:
        mlflow.sklearn.log_model(
            best_model, artifact_path="model",
            signature=signature, input_example=X_input_df,
        )

    new_run_id = run.info.run_id

print(f"  ✓ Re-logged with signature — run {new_run_id}")

# ═══════════════════════════════════════════════════════════════
# §3. REGISTER MODEL IN UNITY CATALOG
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§3. Register Model")
print(f"{'─' * 70}")

client = MlflowClient()
model_name = f"{CATALOG}.{GOLD}.btcusdt_direction_classifier"
model_uri  = f"runs:/{new_run_id}/model"

print(f"  Model name: {model_name}")
print(f"  Model URI:  {model_uri}")

result = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"  ✓ Registered — version {result.version}")

# ═══════════════════════════════════════════════════════════════
# §4. ADD DESCRIPTION & TAGS
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§4. Add Metadata")
print(f"{'─' * 70}")

description = (
    f"Binary classifier for BTCUSDT 1-min direction prediction. "
    f"Best model: {best_model_name}. "
    f"Test ROC-AUC: {auc_val:.4f}, Accuracy: {acc_val:.1%}. "
    f"Trained on Feb 2026 data with {len(feature_cols)} technical indicator features. "
    f"NOTE: Model does NOT beat 'Last TB Label' baseline (AUC 0.708)."
)
client.update_registered_model(name=model_name, description=description)
print(f"  ✓ Description set")

tags = {
    "task":            "binary_classification",
    "symbol":          "BTCUSDT",
    "bar_interval":    "1min",
    "best_model_type": best_model_name,
    "test_roc_auc":    f"{auc_val:.4f}",
    "test_accuracy":   f"{acc_val:.4f}",
    "beats_baseline":  str(auc_val > 0.708),
    "n_features":      str(len(feature_cols)),
    "training_period": "2026-02",
}
for key, value in tags.items():
    client.set_registered_model_tag(name=model_name, key=key, value=value)
    print(f"  Tag: {key} = {value}")

# ═══════════════════════════════════════════════════════════════
# §5. SET ALIAS
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§5. Set Alias")
print(f"{'─' * 70}")

client.set_registered_model_alias(
    name=model_name, alias=alias, version=result.version,
)
print(f"  ✓ Alias '{alias}' → version {result.version}")

# ═══════════════════════════════════════════════════════════════
# §6. VERIFY REGISTRATION
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§6. Verify Registration")
print(f"{'─' * 70}")

model_info = client.get_registered_model(model_name)
latest = client.get_model_version(model_name, result.version)

print(f"  Model:       {model_info.name}")
print(f"  Description: {model_info.description[:80]}...")
print(f"  Version:     {latest.version}")
print(f"  Status:      {latest.status}")
print(f"  Source:      {latest.source}")

# ═══════════════════════════════════════════════════════════════
# §7. QUICK INFERENCE TEST
# ═══════════════════════════════════════════════════════════════
print(f"\n{'─' * 70}")
print("§7. Quick Inference Test")
print(f"{'─' * 70}")

loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}@{alias}")

sample_df = pd.DataFrame(
    X_test_scaled[:5] if use_scaled else X_test[:5],
    columns=feature_cols,
)
preds = loaded_model.predict(sample_df)

print(f"  Sample predictions (first 5 test rows):")
for i, (p, actual) in enumerate(zip(preds, y_test[:5])):
    status = "✓" if int(p) == int(actual) else "✗"
    label_pred = "UP" if int(p) == 1 else "DOWN"
    label_actual = "UP" if int(actual) == 1 else "DOWN"
    print(f"    [{i}] Predicted: {label_pred}  Actual: {label_actual}  {status}")

# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("PHASE 8B SUMMARY")
print("=" * 70)
print(f"  Model registered: {model_name}")
print(f"  Version:          {result.version}")
print(f"  Alias:            {alias}")
print(f"  Model type:       {best_model_name}")
print(f"  Test ROC-AUC:     {auc_val:.4f}")
print(f"  Test Accuracy:    {acc_val:.1%}")
print(f"\n  NEXT STEPS:")
print(f"    1. Improve features to beat baseline (need AUC > 0.708)")
print(f"    2. Try longer prediction horizons (5-min, 15-min)")
print(f"    3. Set up model serving endpoint")
print(f"    4. Create monitoring dashboard for drift")
print(f"    5. Backtest with simulated trading strategy")
print(f"\n{'=' * 70}")
print("✓ Phase 8B complete — Model registered in Unity Catalog")
print(f"{'=' * 70}")